# MiniTcl: análisis manual de scripts y análisis de expresiones con SLY
## Especificación y cuaderno de trabajo del estudiante

**Asignatura:** Compiladores · **Implementación:** Python 3.10 o posterior · **Enfoque:** orientación a objetos y `dataclasses`.

**Objetivo:** construir un lexer manual y un parser descendente recursivo para scripts MiniTcl, y un lexer con `sly.Lexer` y un parser LALR(1) con `sly.Parser` para las expresiones de `expr`. Ambos producen AST y no ejecutan el programa durante el análisis.

| Parte | Lexer | Parser |
|---|---|---|
| Scripts, comandos, palabras y cuerpos | Manual | `ScriptParser`, descendente recursivo |
| Expresiones de `expr` y condiciones de `if`, `while`, `for` | `ExprLexer(sly.Lexer)` | `ExprParser(sly.Parser)` |

Se entrega la especificación, el AST y esqueletos mínimos. **El estudiante completa las reglas a partir de la gramática.** El parser manual suministrado acepta solo palabras de un `TEXT`; el arranque de SLY acepta solo un número. No se entrega un intérprete ni un analizador completo. Las actividades se identifican con `TODO` o `NotImplementedError`; sus pruebas se activan por etapas.

El cuaderno se puede leer y ejecutar sin SLY: la demostración opcional de SLY se omite si no está instalado. Para desarrollar esa parte, instale `sly` en el entorno del kernel, por ejemplo con `%pip install sly` en una celda aparte, y reinicie el kernel si es necesario.

> **Estado de esta copia: actividades completadas.**
>
> Organización de la entrega:
>
> | Módulo | Contenido |
> |---|---|
> | `minitcl/model.py` | AST, `Token`, `ParseError` ampliado (línea, columna, esperado, encontrado, apertura), `TokenCursor`, `cover` |
> | `minitcl/script_lexer.py` | Lexer manual contextual (modos script / comillas / sustitución / llaves) |
> | `minitcl/script_parser.py` | `ScriptParser` descendente recursivo, fachada `parse_minitcl`, callback `read_substitution` |
> | `minitcl/expr_lexer.py` | `ExprLexer(sly.Lexer)` |
> | `minitcl/expr_parser.py` | `ExprParser(sly.Parser)` LALR(1), fachada `parse_minitcl_expr` |
> | `minitcl/list_reader.py` | `parse_list` / `format_list` |
> | `minitcl/command_analyzer.py` | Validación de formas de comandos y reanálisis de cuerpos y condiciones |
> | `tests/` | Pruebas `pytest` positivas y negativas (`python -m pytest -q`) |
> | `examples/` | Programas de aceptación de la sección 15 |
>
> Las celdas `%%writefile` de este cuaderno **son** el código de los módulos: al ejecutarlas se regeneran los archivos idénticos. Las celdas siguientes los importan.

In [1]:
%pip install -q sly

Note: you may need to restart the kernel to use updated packages.


In [2]:
%%writefile minitcl/__init__.py
"""MiniTcl educativo 1.1: lexer manual + parser descendente recursivo de
scripts, y lexer/parser SLY (LALR(1)) para expresiones."""
from .model import *  # noqa: F401,F403
from .model import ParseError, show_ast
from .script_lexer import ScriptLexer, tokenize_script
from .script_parser import ScriptParser, parse_minitcl, read_substitution
from .expr_lexer import ExprLexer
from .expr_parser import ExprParser, parse_minitcl_expr
from .list_reader import parse_list, format_list
from .command_analyzer import (AnalysisResult, analyze_command, analyze_script,
                               analyze_source)

__all__ = [
    "ParseError", "show_ast", "ScriptLexer", "tokenize_script", "ScriptParser",
    "parse_minitcl", "read_substitution", "ExprLexer", "ExprParser",
    "parse_minitcl_expr", "parse_list", "format_list", "AnalysisResult",
    "analyze_command", "analyze_script", "analyze_source",
]

Overwriting minitcl/__init__.py


## 1. Alcance: un MiniTcl útil y realizable

La referencia pequeña es **Picol de antirez**, especialmente su enfoque didáctico; no se pretende comparar con todos los proyectos derivados que también se llaman Picol. Jim Tcl sirve como referencia de una implementación mucho más amplia. El tamaño de la gramática por sí solo no mide la capacidad de un Tcl: muchas ampliaciones son comandos incorporados, sin cambiar el parser de scripts.

Esta propuesta se llama **MiniTcl educativo 1.1**. Amplía el trabajo previo del curso con `for`, `foreach`, listas, diccionarios, operaciones de cadenas, arreglos de índice sencillo y una separación explícita entre AST estructural y validación de comandos.

| Obligatorio en el proyecto completo | Límite para mantener el proyecto realizable |
|---|---|
| Variables escalares y arreglos asociativos | Índices literales simples; sin sustituciones dentro del índice |
| Palabras sin agrupar, entre comillas y entre llaves | Sin expansión `{*}` ni concatenación después de cerrar una palabra agrupada |
| Interpolación y sustituciones `[...]` anidadas | Sin `eval`, `uplevel` ni generación dinámica de cuerpos de control |
| `expr`, condicionales, ciclos y procedimientos recursivos | Cuerpos y condiciones de control escritos entre llaves |
| Listas, diccionarios y operaciones sencillas de cadenas | Sin repertorio completo de Tcl ni bibliotecas de Jim Tcl |
| Ámbitos locales y declaración `global` | Sin namespaces, cierres, objetos ni trazas |
| `catch` y `error` | Sin opciones avanzadas de `return` ni de `catch` |

No se exigen GUI, sockets, procesos, archivos, extensiones nativas, bytecode ni generación para 6502 en esta entrega. El AST podrá servir después como entrada del checker, del intérprete y del generador de código.

**Tareas posibles:** factorial y Fibonacci recursivos; estadísticas de una lista; búsqueda lineal; histograma con diccionarios; tablas de multiplicar; procesamiento sencillo de texto y manejo de errores.

## 2. La decisión central: dos gramáticas y dos momentos

El parser estructural reconoce **scripts, comandos y palabras**. `if`, `set` y `proc` son nombres de comandos; no se convierten en palabras reservadas del lexer.

1. **Análisis estructural:** `Script → Command → Word → Part`. Las llaves producen una palabra literal; los corchetes contienen otro `Script`.
2. **Validación de comandos:** una pasada posterior comprueba aridad y las formas de los incorporados. En cuerpos y condiciones estáticos puede construir vistas `Script` y `Expr` adicionales.
3. **Ejecución futura:** resuelve variables, concatena partes de palabras y ejecuta comandos. Este paso no forma parte del trabajo del parser.

En `set mensaje {if algo}`, el bloque es texto; **no** es un `If`. En `while {$i < 10} {incr i}`, los argumentos son palabras literales que el comando interpreta como expresión y script, respectivamente.

Se recomienda conservar el AST estructural y asociarle un mapa de cuerpos y expresiones ya analizados. Un AST especializado con `If` o `While` puede añadirse después, pero no debe cambiar el significado de cualquier palabra entre llaves.

### Respecto de la especificación anterior del curso

Se conserva la precedencia de `expr`, incluido `-2 ** 2 = -(2 ** 2)`. Se precisan tres cambios: `#` abre comentario solo al comienzo de un comando; una palabra puede contener varias partes (`hola_$nombre`); y se conservan espacios significativos para distinguir palabras. No debe mezclarse este contrato con el lexer anterior sin adaptarlo.

### Frontera entre las dos herramientas

`ScriptParser` usa exclusivamente los tokens del lexer manual. El contenido literal de un argumento de `expr` se entrega a `parse_minitcl_expr()`, que usa `ExprLexer` y `ExprParser` de SLY. Las condiciones de `if`, `elseif`, `while` y `for` usan exactamente la misma fachada. Los cuerpos, la inicialización y el paso de `for` vuelven al parser manual.

En una expresión, el token `CMD_SUBST` guarda un `Script` construido por el parser manual al leer `[...]`. El parser SLY lo convierte en `SubstitutionExpr`; no ejecuta ese script. Por tanto, el cortocircuito sigue siendo posible.


## 3. Notación y reglas léxicas

En EBNF, `{ X }` significa repetición y `[ X ]` significa opcional. Los caracteres **entre comillas** son terminales: `"{"` representa una llave del programa, no una repetición de la notación. `ε` denota vacío.

### Contrato del lexer de scripts

El lexer debe conservar los límites de palabras. No basta con descartar todos los espacios.

| Token | Contenido / comportamiento |
|---|---|
| `TEXT` | Fragmento literal ya decodificado; incluye escapes procesados fuera de llaves |
| `VAR` | Nombre escalar; en un arreglo conserva también el índice literal |
| `BRACED` | Una palabra completa entre llaves; valor interior literal y span de toda la palabra |
| `QOPEN`, `QCLOSE` | Inicio y fin de palabra entre comillas |
| `LBRACKET`, `RBRACKET` | Abren y cierran una sustitución de comando |
| `WS` | Uno o más espacios horizontales fuera de palabras agrupadas |
| `SEP` | `;` o salto de línea fuera de llaves y comillas |
| `EOF` | Fin de la fuente, siempre presente |

`BRACED` se obtiene contando llaves anidadas; una expresión regular aislada no resuelve el balanceo general. Sus comillas y corchetes son texto. `\{` y `\}` no cambian la profundidad, y se conservan con su barra dentro de llaves. Dentro de comillas, espacios, `;` y saltos se incorporan a `TEXT`; `$` y `[` sí abren sustituciones. Una sustitución dentro de comillas vuelve temporalmente al modo script y luego restaura el modo comillas.

`{` y `"` agrupan solo al inicio de una palabra. Para simplificar, en una palabra no agrupada se rechazan llaves y comillas no escapadas; escribirlas mediante escapes. Tras cerrar una palabra agrupada solo puede seguir `WS`, `SEP`, `RBRACKET` o `EOF`.

`#` comienza un comentario solo donde podría empezar **un comando**, después de espacios iniciales. El comentario termina en el salto de línea; ese salto sigue separando comandos. `puts a#b` y `puts #dato` contienen texto, no comentarios. Al entrar en `[` empieza un nuevo contexto de comando.

Escapes obligatorios fuera de llaves: `\n`, `\t`, `\r`, `\\`, `\ `, `\$`, `\[`, `\]`, `\{`, `\}`, `\"`, `\;`, `\#`. Un escape desconocido es error en este perfil. Se normalizan CRLF/CR a LF antes del lexer; los offsets se refieren a esa fuente normalizada. **La continuación barra + salto de línea queda fuera del perfil obligatorio** y debe rechazarse explícitamente; puede agregarse como ampliación con mapa de posiciones.

### Variables y números

```ebnf
identifier = letter_or_underscore, { letter_or_underscore | digit } ;
variable   = "$", identifier, [ "(", index, ")" ]
           | "${", identifier, "}" ;
index      = index_char, { index_char } ;
```

Los identificadores usan letras ASCII. `index_char` es letra, dígito, `_`, `.`, `-` o `:`. Se aceptan `$a(3)` y `$a(nombre)`; se excluyen `$a($i)`, índices vacíos y arreglos multidimensionales. `${x}` desambigua `"${x}_fin"`. Un `$` sin nombre válido es error; para imprimirlo se escribe `\$`.

En scripts, `25` sigue siendo texto. Solo el lexer de expresiones genera `NUMBER`: `(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?`. El signo inicial es un operador; no pertenece al número. No hay hexadecimales ni números complejos en este perfil.

## 4. Gramática estructural de MiniTcl

Esta gramática opera sobre los tokens anteriores. El lexer no crea un `TEXT` vacío. `BRACED` puede tener contenido vacío y las comillas pueden delimitar una cadena vacía.

```ebnf
program       = script, EOF ;
script        = { WS | SEP }, [ command, { command_gap, command } ],
                { WS | SEP } ;
command_gap   = { WS }, SEP, { WS | SEP } ;
command       = word, { WS, word } ;
word          = BRACED | quoted_word | bare_word ;
quoted_word   = QOPEN, { part }, QCLOSE ;
bare_word     = part, { part } ;
part          = TEXT | VAR | substitution ;
substitution  = LBRACKET, script, RBRACKET ;
```

**Convención operativa para los espacios finales:** en `command`, después de consumir `WS`, solo se lee otra palabra si el siguiente token inicia `word`; si es un separador o el fin del contexto, termina el comando. El EBNF describe la secuencia válida; esta regla resuelve la decisión sobre el espacio sin retroceso.

`script` termina ante `EOF` en el nivel exterior o ante `RBRACKET` dentro de una sustitución. El método que abrió `[` consume el `RBRACKET`; el script interior lo deja pendiente. Un `RBRACKET` en el nivel exterior es error.

### ¿Por qué no usar `split()`?

```tcl
puts "Hola $nombre: [expr {2 + 3}]"
set dato {una palabra con espacios}
set ruta prefijo_${nombre}_fin
```

El primer comando tiene dos palabras, el segundo tres y el tercero tres. Los espacios dentro de agrupaciones no separan argumentos; las partes contiguas de una palabra se concatenan **al ejecutar**, sin crear nuevos argumentos.

### Recursión que sí es necesaria

`parse_substitution()` llama a `parse_script()`; este puede encontrar otra sustitución. Eso es descenso recursivo válido. No es recursividad izquierda: antes de la llamada se consumió `LBRACKET`.

## 5. Gramática de expresiones para SLY, con AST

Esta parte se implementa con **`sly.Lexer` y `sly.Parser`**, que construye un analizador LALR(1). Se utiliza una gramática por niveles, con recursividad izquierda para los operadores asociativos a la izquierda. No debe traducirse a métodos descendentes recursivos.

```ebnf
expression     = logical_or ;
logical_or     = logical_or, "||", logical_and | logical_and ;
logical_and    = logical_and, "&&", equality | equality ;
equality       = equality, ("==" | "!=" | "eq" | "ne"), relational | relational ;
relational     = relational, ("<" | "<=" | ">" | ">="), additive | additive ;
additive       = additive, ("+" | "-"), multiplicative | multiplicative ;
multiplicative = multiplicative, ("*" | "/" | "%"), unary | unary ;
unary          = ("+" | "-" | "!"), unary | power ;
power          = primary, "**", unary | primary ;
primary        = NUMBER | BOOL | VAR | STRING
               | "(", expression, ")" | CMD_SUBST ;
```

SLY recibe el fin de entrada al agotarse el iterador de tokens: **no se crea un token `EOF` para SLY**. Su símbolo inicial será `expression`; debe rechazar residuos como `2 3`. El lexer manual y su cursor sí conservan el `EOF` definido para scripts.

Los niveles de esta gramática ya definen precedencia y asociatividad. No hace falta una tabla `precedence` adicional. Si el estudiante opta por una gramática plana con tabla de precedencia, deberá justificarla y conservar exactamente los AST de las pruebas, especialmente la relación entre unarios y potencia.

| Grafía / categoría | Nombre de token recomendado para SLY |
|---|---|
| `||`, `&&` | `OR`, `AND` |
| `==`, `!=`, `eq`, `ne` | `EQ`, `NE`, `STREQ`, `STRNE` |
| `<`, `<=`, `>`, `>=` | `LT`, `LE`, `GT`, `GE` |
| `+`, `-`, `*`, `/`, `%`, `**`, `!` | `PLUS`, `MINUS`, `TIMES`, `DIVIDE`, `MOD`, `POWER`, `NOT` |
| `(`, `)` | `LPAREN`, `RPAREN` |
| Número, booleano, variable y cadena literal | `NUMBER`, `BOOL`, `VAR`, `STRING` |
| Sustitución de comando balanceada | `CMD_SUBST`, cuyo `value` será un `Script` |

El lexer de SLY debe definir los patrones largos antes de los cortos: `**` antes de `*`, `<=` antes de `<`, etc. `eq`, `ne` y los booleanos se reconocen como palabras completas. Los blancos fuera de cadenas se ignoran en el lexer de expresiones, conservando los datos necesarios para calcular posiciones. No se incorporan comentarios de script dentro de `expr`.

`BOOL` acepta `true`, `false`, `yes`, `no`, `on`, `off`; `0` y `1` son números convertibles a booleano. `STRING` es una cadena entre comillas literal, sin interpolación en este perfil; rechazar `$` o `[` no escapados dentro de ella. No hay cadenas desnudas, funciones matemáticas, ternario ni operaciones de bits en la entrega obligatoria.

### Contrato para `CMD_SUBST`

Al encontrar `[`, una acción del lexer SLY delega la lectura balanceada al analizador manual mediante un callback. Este devuelve el AST del script y el índice inmediatamente posterior al `]`. La acción devuelve un único token `CMD_SUBST` y actualiza `self.index` para continuar después del cierre. No se busca simplemente el siguiente `]` ni se usa una expresión regular para balancear todo el script. Debe respetarse el anidamiento y los modos de llaves/comillas del lexer manual.

Firma sugerida del callback:

```python
read_substitution(text, opening_index, base_offset, source_name) -> tuple[Script, int]
```

`opening_index` y el índice devuelto son locales a `text`; los spans del AST se construyen sumando `base_offset`. Al avanzar sobre una sustitución multilínea, el lexer SLY también actualiza su contador de líneas. Cada llamada anidada a la fachada de expresiones crea sus propias instancias de lexer/parser, para no sobrescribir un análisis en curso.

| Entrada | Forma del AST esperada |
|---|---|
| `2 + 3 * 4` | `Binary('+', 2, Binary('*', 3, 4))` |
| `10 - 3 - 2` | `Binary('-', Binary('-', 10, 3), 2)` |
| `2 ** 3 ** 2` | `Binary('**', 2, Binary('**', 3, 2))` |
| `-2 ** 2` | `Unary('-', Binary('**', 2, 2))` |
| `2 ** -3` | `Binary('**', 2, Unary('-', 3))` |
| `$x != 0 && 10 / $x > 1` | `Binary('&&', comparación, comparación)` |

Las comparaciones encadenadas se agrupan por la izquierda. Las acciones de SLY construyen nodos, no calculan resultados. El evaluador posterior hace el cortocircuito; no usar `eval()` de Python.

## 6. Contrato de comandos: formas que debe validar la siguiente pasada

En esta tabla, `word` representa **una sola palabra** del comando, `body` es una palabra `BRACED` reinterpretada como script y `test` es una palabra `BRACED` reinterpretada como expresión. El estudiante implementa el parser genérico primero; estas formas no son alternativas del lexer.

```ebnf
set_cmd      = "set", name, [ word ] ;
unset_cmd    = "unset", name ;
incr_cmd     = "incr", name, [ word ] ;
append_cmd   = "append", name, word, { word } ;
puts_cmd     = "puts", [ "-nonewline" ], word ;
expr_cmd     = "expr", test ;
if_cmd       = "if", test, [ "then" ], body,
               { "elseif", test, [ "then" ], body }, [ "else", body ] ;
while_cmd    = "while", test, body ;
for_cmd      = "for", body, test, body, body ;
foreach_cmd  = "foreach", identifier, word, body ;
proc_cmd     = "proc", identifier, parameters, body ;
parameters   = BRACED ;   (* su contenido: lista plana de identificadores únicos *)
return_cmd   = "return", [ word ] ;
break_cmd    = "break" ;
continue_cmd = "continue" ;
global_cmd   = "global", identifier, { identifier } ;
catch_cmd    = "catch", body, [ identifier ] ;
error_cmd    = "error", word ;
call_cmd     = word, { word } ;
```

`call_cmd` es la forma estructural general y no una alternativa que deba probarse después de fallar con un incorporado. `name` es un nombre escalar o una referencia escrita `a(indice)` con el mismo índice simple del léxico. En `set a(3) 8` no se escribe `$` en el nombre que recibe el comando.

**Perfil estático del curso:** nombres de incorporados no redefinibles; nombres de procedimientos y variables receptoras de asignación escritos literalmente; condiciones y cuerpos entre llaves; `expr` recibe exactamente una expresión entre llaves. Los nombres de comandos calculados son válidos estructuralmente pero el validador los rechaza como fuera del perfil. `set accion puts; $accion hola` ilustra esa diferencia. Los argumentos de una llamada sí pueden usar interpolación y sustituciones.

Los parámetros no tienen valores por defecto ni `args` variádico. `foreach` tiene exactamente una variable y una lista; no se exigen formas de múltiples listas. `global` se permite dentro de procedimientos y enlaza un nombre con el entorno global.

### Biblioteca mínima de datos

| Forma | Semántica que deberá tener el intérprete futuro |
|---|---|
| `list ?valor ...?` | Construye una representación de lista que preserve cada elemento |
| `llength lista` | Número de elementos |
| `lindex lista indice` | Índice entero base 0; fuera de rango devuelve cadena vacía |
| `lappend nombre ?valor ...?` | Añade elementos; crea lista vacía si no existe la variable |
| `lset nombre indice valor` | Reemplaza un elemento existente; índice fuera de rango es error |
| `dict create ?clave valor ...?` | Diccionario serializado como pares; claves repetidas conservan el último valor |
| `dict get diccionario clave` | Devuelve valor; clave ausente es error |
| `dict exists diccionario clave` | Devuelve `0` o `1` |
| `dict set nombre clave valor` | Modifica variable diccionario; la crea si no existe |
| `string length texto` | Número de caracteres Unicode |
| `string tolower texto` / `string toupper texto` | Conversión de mayúsculas/minúsculas |
| `string compare a b` | Devuelve `-1`, `0` o `1`, orden lexicográfico por puntos de código |
| `split texto ?separadores?` | Separa por cualquiera de los caracteres dados; por defecto espacio, tabulador y salto |
| `join lista ?separador?` | Une elementos; separador por defecto un espacio |

En `split`, separadores contiguos producen elementos vacíos; un separador vacío divide en caracteres. En `dict create` se exige un número par de argumentos. Los arreglos `a(k)` son variables asociativas; los diccionarios son valores. No se necesita un comando `array` para la entrega obligatoria.

Estas operaciones amplían las tareas posibles sin ampliar la gramática estructural del lenguaje.

## 7. Las listas requieren su propio lector

El contenido de una lista no es un script: `$x`, `[comando]` y `;` son datos, sin sustituciones ni separación de comandos. El salto de línea separa elementos, igual que un espacio.

```ebnf
list_value    = { LIST_WS }, [ element, { LIST_WS, element } ], { LIST_WS } ;
element       = LIST_BRACED | LIST_QUOTED | LIST_BARE ;
```

`LIST_WS` contiene uno o más blancos. El lector de listas balancea llaves, elimina agrupaciones y procesa los escapes admitidos fuera de llaves; no evalúa variables ni comandos. Tras una agrupación exige blanco o fin. En elementos no agrupados se rechazan llaves y comillas no escapadas, como simplificación coherente con el script.

Contrato esencial: `parse_list(format_list(values)) == values`. No construir listas mediante `' '.join(values)` cuando los elementos contienen espacios, llaves o cadenas vacías. El lector de parámetros de `proc` puede reutilizar el lector de listas y después exigir identificadores únicos. No se entrega su implementación.

## 8. Modelo del AST

Se entrega el modelo de datos para que la actividad se concentre en traducir la gramática. `Span` usa offsets semiabiertos `[start, end)` en la fuente normalizada e incluye el nombre de la fuente. En `BRACED`, el span de `Literal` cubre el interior y el de `Word` incluye las llaves.

| Nodo | Responsabilidad |
|---|---|
| `Script` | Secuencia de comandos |
| `Command` | Secuencia de palabras; la primera es el nombre del comando |
| `Word` | Conserva modo `bare`, `quoted` o `braced` y sus partes |
| `Literal` | Texto que no debe resolverse como variable o comando |
| `Variable` | Referencia escalar o de arreglo con índice literal |
| `CommandSubstitution` | Script cuyo resultado será parte de una palabra |
| `Constant`, `VariableExpr`, `SubstitutionExpr` | Hojas del AST de expresiones |
| `Unary`, `Binary` | Operaciones sin evaluar |

Una palabra entre llaves tiene exactamente un `Literal`, incluso si es vacío. Una palabra entre comillas vacía puede tener cero partes. Los espacios de separación y los paréntesis de agrupación no necesitan nodos propios.

In [3]:
%%writefile minitcl/model.py
"""Modelo del AST, tokens, cursor y error de análisis de MiniTcl.

Las clases se conservan con el mismo contrato que entrega el cuaderno
(sección 8 y 9). ``ParseError`` se amplía de forma compatible para
transportar el elemento esperado, el encontrado, la posición de apertura
de una agrupación sin cerrar y, cuando se conoce la fuente, la línea y la
columna del error.
"""
from __future__ import annotations

from dataclasses import dataclass, fields, is_dataclass
from typing import Literal as Mode


# ---------------------------------------------------------------------------
# Posiciones
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Span:
    start: int
    end: int
    source: str = "<input>"


@dataclass(frozen=True, kw_only=True)
class Node:
    span: Span


# ---------------------------------------------------------------------------
# AST estructural de scripts
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Literal(Node):
    text: str


@dataclass(frozen=True)
class Variable(Node):
    name: str
    index: str | None = None


@dataclass(frozen=True)
class CommandSubstitution(Node):
    script: Script


Part = Literal | Variable | CommandSubstitution


@dataclass(frozen=True)
class Word(Node):
    parts: tuple[Part, ...]
    mode: Mode["bare", "quoted", "braced"] = "bare"


@dataclass(frozen=True)
class Command(Node):
    words: tuple[Word, ...]


@dataclass(frozen=True)
class Script(Node):
    commands: tuple[Command, ...]


# ---------------------------------------------------------------------------
# AST de expresiones
# ---------------------------------------------------------------------------

@dataclass(frozen=True, kw_only=True)
class Expr(Node):
    pass


@dataclass(frozen=True)
class Constant(Expr):
    value: int | float | str | bool


@dataclass(frozen=True)
class VariableExpr(Expr):
    variable: Variable


@dataclass(frozen=True)
class SubstitutionExpr(Expr):
    script: Script


@dataclass(frozen=True)
class Unary(Expr):
    operator: str
    operand: Expr


@dataclass(frozen=True)
class Binary(Expr):
    operator: str
    left: Expr
    right: Expr


# ---------------------------------------------------------------------------
# Tokens, errores y cursor
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Token:
    kind: str
    value: str
    span: Span
    index: str | None = None


def line_column(text: str, offset: int) -> tuple[int, int]:
    """Línea y columna (desde 1) de ``offset`` en la fuente normalizada."""
    offset = max(0, min(offset, len(text)))
    line = text.count("\n", 0, offset) + 1
    line_start = text.rfind("\n", 0, offset) + 1
    return line, offset - line_start + 1


class ParseError(Exception):
    """Error de entrada MiniTcl (léxico, sintáctico o de validación).

    Compatible con la firma original ``ParseError(message, span)``.
    """

    def __init__(self, message: str, span: Span, *, expected: str | None = None,
                 found: str | None = None, opening: Span | None = None,
                 stage: str = "syntax"):
        self.message = message
        self.span = span
        self.expected = expected
        self.found = found
        self.opening = opening
        self.stage = stage          # "lexer", "syntax", "expr" o "validation"
        self.line: int | None = None
        self.column: int | None = None
        self.snippet: str | None = None
        super().__init__(self._headline())

    def _headline(self) -> str:
        where = f"{self.span.source}:"
        if self.line is not None:
            where += f"{self.line}:{self.column}"
        else:
            where += f"{self.span.start}"
        text = f"{where}: {self.message}"
        if self.expected is not None or self.found is not None:
            text += f" (esperado: {self.expected or '-'}; encontrado: {self.found or '-'})"
        return text

    def attach_source(self, text: str) -> "ParseError":
        """Calcula línea, columna y fragmento a partir de la fuente normalizada."""
        if self.line is not None:
            return self
        self.line, self.column = line_column(text, self.span.start)
        line_start = text.rfind("\n", 0, self.span.start) + 1
        line_end = text.find("\n", self.span.start)
        if line_end < 0:
            line_end = len(text)
        source_line = text[line_start:line_end]
        width = max(1, min(self.span.end, line_end) - self.span.start)
        caret = " " * (self.span.start - line_start) + "^" * width
        self.snippet = f"{source_line}\n{caret}"
        if self.opening is not None:
            open_line, open_col = line_column(text, self.opening.start)
            self.snippet += f"\n(apertura en línea {open_line}, columna {open_col})"
        self.args = (self._headline(),)
        return self

    def report(self) -> str:
        """Mensaje completo con fragmento, si se adjuntó la fuente."""
        return self._headline() + (f"\n{self.snippet}" if self.snippet else "")


class TokenCursor:
    def __init__(self, tokens):
        self.tokens = tuple(tokens)
        if not self.tokens or self.tokens[-1].kind != "EOF":
            raise ValueError("El lexer debe agregar EOF")
        self.position = 0

    @property
    def current(self):
        return self.tokens[self.position]

    def accept(self, kind):
        if self.current.kind != kind:
            return None
        token = self.current
        if kind != "EOF":
            self.position += 1
        return token

    def expect(self, kind):
        token = self.accept(kind)
        if token is None:
            raise ParseError(
                f"Se esperaba {kind}; se encontró {self.current.kind}",
                self.current.span, expected=kind, found=self.current.kind)
        return token

    def skip(self, *kinds):
        while self.current.kind in kinds and self.current.kind != "EOF":
            self.position += 1


def cover(first: Span, last: Span):
    if first.source != last.source:
        raise ValueError("No combinar posiciones de fuentes diferentes")
    return Span(first.start, last.end, first.source)


def normalize_newlines(text: str) -> str:
    """CRLF y CR se convierten en LF antes de cualquier análisis."""
    return text.replace("\r\n", "\n").replace("\r", "\n")


def show_ast(value, indent=0):
    """Impresión estructurada sin Graphviz ni dependencias externas."""
    prefix = " " * indent
    if is_dataclass(value):
        print(prefix + type(value).__name__)
        for field in fields(value):
            if field.name == "span":
                continue
            item = getattr(value, field.name)
            print(prefix + "  " + field.name + ":")
            show_ast(item, indent + 4)
    elif isinstance(value, tuple):
        for item in value:
            show_ast(item, indent)
    else:
        print(prefix + repr(value))

Overwriting minitcl/model.py


In [4]:
from minitcl.model import (Span, Node, Literal, Variable, CommandSubstitution, Part, Word,
                           Command, Script, Expr, Constant, VariableExpr, SubstitutionExpr,
                           Unary, Binary)
from dataclasses import fields, is_dataclass
print("Modelo del AST importado desde minitcl/model.py")

Modelo del AST importado desde minitcl/model.py


### Un AST construido a mano para comprender el resultado

La siguiente celda **no analiza texto**. Muestra el AST que se espera obtener para `puts hola`. Observe que no se imprime `hola` como efecto del programa: únicamente se inspecciona su estructura.

In [5]:
def show_ast(value, indent=0):
    """Impresión estructurada sin Graphviz ni dependencias externas."""
    prefix = " " * indent
    if is_dataclass(value):
        print(prefix + type(value).__name__)
        for field in fields(value):
            if field.name == "span":
                continue
            item = getattr(value, field.name)
            print(prefix + "  " + field.name + ":")
            show_ast(item, indent + 4)
    elif isinstance(value, tuple):
        for item in value:
            show_ast(item, indent)
    else:
        print(prefix + repr(value))

example_ast = Script(
    commands=(Command(words=(
        Word(parts=(Literal(text="puts", span=Span(0, 4)),), span=Span(0, 4)),
        Word(parts=(Literal(text="hola", span=Span(5, 9)),), span=Span(5, 9)),
    ), span=Span(0, 9)),), span=Span(0, 9))
show_ast(example_ast)

Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'puts'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'hola'
          mode:
            'bare'


## 9. Infraestructura mínima: tokens, cursor y diagnósticos

El lexer deberá producir la secuencia de `Token`. No se entrega un lexer completo. Para `VAR`, `value` contiene el nombre e `index` el índice opcional; para `BRACED`, `value` es el contenido interior sin decodificar. La implementación posterior debe conservar la fuente original normalizada para reanalizar los cuerpos con offsets correctos.

El cursor permite mirar, aceptar o exigir tokens. Los métodos del estudiante deben producir `ParseError` para entradas incorrectas; `NotImplementedError` significa una actividad todavía pendiente, no un error del programa MiniTcl.

In [6]:
# Token, ParseError, TokenCursor y cover están definidos en minitcl/model.py (celda anterior).
# ParseError conserva la firma ParseError(message, span) y añade expected, found,
# opening, stage, line y column.
from minitcl.model import Token, ParseError, TokenCursor, cover, normalize_newlines, line_column

## 10. Esqueleto del parser: completar desde la gramática

Solo se resuelve el caso de una palabra que consta de **un `TEXT`**. Se deja implementada la coordinación básica de scripts y comandos para mostrar el uso del cursor. El estudiante debe ampliar `parse_word`, desarrollar los métodos pendientes y escribir el lexer.

El argumento `stop` fija el fin del contexto, pero no consume ese token. En particular, el analizador exterior exige `EOF` y el de sustitución deberá exigir `RBRACKET`. Cada iteración de un bucle tiene que consumir al menos un token o terminar: un error habitual es quedar atrapado en un bucle infinito.

### Lexer manual de scripts

El lexer es una clase con un método por modo. Los cambios de contexto se implementan con llamadas recursivas (`lex_bracket` → `lex_script` anidado → `lex_quoted` …), de modo que la pila de llamadas es la pila de modos y al salir de una sustitución se restaura automáticamente el modo comillas o script. `BRACED` se obtiene contando profundidad, no con una expresión regular.

In [7]:
%%writefile minitcl/script_lexer.py
"""Lexer manual de scripts MiniTcl (sección 3 del cuaderno).

El lexer es contextual: el significado de un carácter depende del modo en
que aparece.

* **modo script**: ``WS`` y ``SEP`` separan palabras y comandos; ``{`` y
  ``"`` agrupan solo al comienzo de una palabra; ``#`` abre un comentario
  solo donde podría empezar un comando.
* **modo comillas**: espacios, ``;`` y saltos de línea son texto; ``$`` y
  ``[`` siguen abriendo sustituciones.
* **modo sustitución**: al entrar en ``[`` empieza un nuevo contexto de
  comando (modo script anidado) que termina en el ``]`` correspondiente;
  al salir se restaura el modo anterior (script o comillas).
* **llaves**: se balancean contando profundidad; el contenido es literal.

Cada modo corresponde a un método; los cambios de contexto son llamadas
recursivas, de modo que la pila de llamadas de Python es la pila de modos.
"""
from __future__ import annotations

import re

from .model import ParseError, Span, Token, normalize_newlines

IDENTIFIER_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
INDEX_CHARS = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_.-:")

ESCAPES = {
    "n": "\n", "t": "\t", "r": "\r", "\\": "\\", " ": " ", "$": "$",
    "[": "[", "]": "]", "{": "{", "}": "}", '"': '"', ";": ";", "#": "#",
}

HSPACE = " \t"
WORD_END = " \t;\n]"          # fin de una palabra no agrupada
AFTER_GROUP = " \t;\n]"       # lo único que puede seguir a una palabra agrupada


def decode_escape(text: str, pos: int, base: int, source: str,
                  stage: str = "lexer") -> tuple[str, int]:
    """Decodifica el escape que empieza en ``text[pos] == '\\'``.

    Devuelve el carácter producido y la posición siguiente. Comparte la
    tabla de escapes con el lexer de expresiones y el lector de listas.
    """
    if pos + 1 >= len(text):
        raise ParseError("Barra invertida al final de la fuente",
                         Span(base + pos, base + pos + 1, source),
                         expected="carácter escapado", found="EOF", stage=stage)
    nxt = text[pos + 1]
    if nxt == "\n":
        raise ParseError("La continuación barra + salto de línea está fuera del perfil",
                         Span(base + pos, base + pos + 2, source),
                         expected="escape admitido", found="\\<salto>", stage=stage)
    if nxt not in ESCAPES:
        raise ParseError(f"Escape desconocido '\\{nxt}'",
                         Span(base + pos, base + pos + 2, source),
                         expected="escape admitido", found=f"\\{nxt}", stage=stage)
    return ESCAPES[nxt], pos + 2


def scan_variable(text: str, pos: int, base: int, source: str,
                  stage: str = "lexer") -> tuple[str, str | None, int]:
    """Analiza ``$nombre``, ``$nombre(indice)`` o ``${nombre}`` desde ``pos``.

    Devuelve ``(nombre, indice, posicion_siguiente)``. Lo reutiliza el
    lexer de expresiones para que ambas herramientas acepten la misma
    sintaxis de variables.
    """
    start = pos
    pos += 1  # '$'
    if pos < len(text) and text[pos] == "{":
        close = text.find("}", pos + 1)
        name = text[pos + 1:close] if close >= 0 else ""
        if close < 0 or not IDENTIFIER_RE.fullmatch(name):
            raise ParseError("Variable '${...}' mal formada",
                             Span(base + start, base + (close + 1 if close >= 0 else len(text)), source),
                             expected="${identificador}", found=text[start:close + 1 if close >= 0 else len(text)],
                             stage=stage)
        return name, None, close + 1
    match = IDENTIFIER_RE.match(text, pos)
    if not match:
        found = text[pos] if pos < len(text) else "EOF"
        raise ParseError("'$' sin nombre de variable válido (use \\$ para un $ literal)",
                         Span(base + start, base + start + 1, source),
                         expected="identificador", found=found, stage=stage)
    name = match.group()
    pos = match.end()
    if pos < len(text) and text[pos] == "(":
        idx_start = pos + 1
        end = idx_start
        while end < len(text) and text[end] in INDEX_CHARS:
            end += 1
        if end >= len(text) or text[end] != ")":
            found = text[end] if end < len(text) else "EOF"
            raise ParseError("Índice de arreglo fuera del perfil: solo se admiten índices literales simples",
                             Span(base + start, base + min(end + 1, len(text)), source),
                             expected="carácter de índice o ')'", found=found, stage=stage)
        if end == idx_start:
            raise ParseError("Índice de arreglo vacío",
                             Span(base + start, base + end + 1, source),
                             expected="índice", found=")", stage=stage)
        return name, text[idx_start:end], end + 1
    return name, None, pos


class ScriptLexer:
    """Convierte una fuente MiniTcl en la secuencia de ``Token`` del contrato."""

    def __init__(self, source: str, source_name: str = "<input>", base_offset: int = 0):
        self.text = normalize_newlines(source)
        self.source_name = source_name
        self.base = base_offset
        self.pos = 0
        self.tokens: list[Token] = []

    # -- utilidades ---------------------------------------------------------
    def span(self, start: int, end: int) -> Span:
        return Span(self.base + start, self.base + end, self.source_name)

    def emit(self, kind: str, value: str, start: int, end: int, index: str | None = None):
        self.tokens.append(Token(kind, value, self.span(start, end), index))

    def error(self, message: str, start: int, end: int | None = None, **kw) -> ParseError:
        return ParseError(message, self.span(start, end if end is not None else start + 1),
                          stage="lexer", **kw)

    def peek(self, offset: int = 0) -> str:
        i = self.pos + offset
        return self.text[i] if i < len(self.text) else ""

    # -- punto de entrada ---------------------------------------------------
    def tokenize(self) -> list[Token]:
        self.lex_script(opening=None)
        self.emit("EOF", "", len(self.text), len(self.text))
        return self.tokens

    def tokenize_substitution(self, opening_index: int) -> list[Token]:
        """Tokens de ``[ ... ]`` desde ``opening_index`` (callback de SLY)."""
        self.pos = opening_index
        self.lex_bracket()
        end = self.pos
        self.emit("EOF", "", end, end)
        return self.tokens

    # -- modo script --------------------------------------------------------
    def lex_script(self, opening: Span | None):
        """Lee comandos hasta EOF (nivel exterior) o hasta ``]`` (anidado).

        En el nivel anidado no consume el ``]``: lo hace ``lex_bracket``.
        """
        at_command_start = True
        text = self.text
        while self.pos < len(text):
            c = text[self.pos]
            if c in HSPACE:
                start = self.pos
                while self.pos < len(text) and text[self.pos] in HSPACE:
                    self.pos += 1
                self.emit("WS", text[start:self.pos], start, self.pos)
            elif c in ";\n":
                self.emit("SEP", c, self.pos, self.pos + 1)
                self.pos += 1
                at_command_start = True
            elif c == "#" and at_command_start:
                # Comentario: termina antes del salto, que sigue separando.
                end = text.find("\n", self.pos)
                self.pos = len(text) if end < 0 else end
            elif c == "]":
                if opening is not None:
                    return
                # Cierre fuera de contexto: se emite y el parser lo rechaza.
                self.emit("RBRACKET", "]", self.pos, self.pos + 1)
                self.pos += 1
                at_command_start = False
            else:
                self.lex_word()
                at_command_start = False
        if opening is not None:
            raise ParseError("Falta ']' que cierre la sustitución de comando",
                             self.span(len(text), len(text)), expected="]", found="EOF",
                             opening=opening, stage="lexer")

    def lex_word(self):
        c = self.text[self.pos]
        if c == "{":
            self.lex_braced()
            self.check_after_group()
        elif c == '"':
            self.lex_quoted()
            self.check_after_group()
        else:
            self.lex_bare()

    def check_after_group(self):
        c = self.peek()
        if c and c not in AFTER_GROUP:
            raise self.error("Texto después de cerrar una palabra agrupada", self.pos,
                             expected="espacio, separador, ']' o fin", found=c)

    # -- llaves -------------------------------------------------------------
    def lex_braced(self):
        text = self.text
        start = self.pos
        depth = 0
        i = start
        while i < len(text):
            ch = text[i]
            if ch == "\\" and i + 1 < len(text):
                i += 2          # \{ y \} no cambian la profundidad
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    self.emit("BRACED", text[start + 1:i], start, i + 1)
                    self.pos = i + 1
                    return
            i += 1
        raise ParseError("Falta '}' que cierre la llave", self.span(len(text), len(text)),
                         expected="}", found="EOF", opening=self.span(start, start + 1),
                         stage="lexer")

    # -- comillas -----------------------------------------------------------
    def lex_quoted(self):
        text = self.text
        start = self.pos
        self.emit("QOPEN", '"', start, start + 1)
        self.pos += 1
        buf: list[str] = []
        buf_start = self.pos

        def flush():
            if buf:
                self.emit("TEXT", "".join(buf), buf_start, self.pos)
                buf.clear()

        while self.pos < len(text):
            c = text[self.pos]
            if c == '"':
                flush()
                self.emit("QCLOSE", '"', self.pos, self.pos + 1)
                self.pos += 1
                return
            if c == "$":
                flush()
                self.lex_variable()
                buf_start = self.pos
            elif c == "[":
                flush()
                self.lex_bracket()       # vuelve temporalmente al modo script
                buf_start = self.pos     # ... y se restaura el modo comillas
            elif c == "\\":
                if not buf:
                    buf_start = self.pos
                ch, self.pos = decode_escape(text, self.pos, self.base, self.source_name)
                buf.append(ch)
            else:
                if not buf:
                    buf_start = self.pos
                buf.append(c)
                self.pos += 1
        raise ParseError('Faltan comillas de cierre \'"\'', self.span(len(text), len(text)),
                         expected='"', found="EOF", opening=self.span(start, start + 1),
                         stage="lexer")

    # -- palabras no agrupadas ---------------------------------------------
    def lex_bare(self):
        text = self.text
        buf: list[str] = []
        buf_start = self.pos

        def flush():
            if buf:
                self.emit("TEXT", "".join(buf), buf_start, self.pos)
                buf.clear()

        while self.pos < len(text) and text[self.pos] not in WORD_END:
            c = text[self.pos]
            if c == "$":
                flush()
                self.lex_variable()
                buf_start = self.pos
            elif c == "[":
                flush()
                self.lex_bracket()
                buf_start = self.pos
            elif c == "\\":
                if not buf:
                    buf_start = self.pos
                ch, self.pos = decode_escape(text, self.pos, self.base, self.source_name)
                buf.append(ch)
            elif c in '{}"':
                raise self.error(f"'{c}' no escapado dentro de una palabra no agrupada",
                                 self.pos, expected="texto o escape", found=c)
            else:
                if not buf:
                    buf_start = self.pos
                buf.append(c)
                self.pos += 1
        flush()

    # -- sustituciones ------------------------------------------------------
    def lex_variable(self):
        start = self.pos
        name, index, end = scan_variable(self.text, start, self.base, self.source_name)
        self.emit("VAR", name, start, end, index)
        self.pos = end

    def lex_bracket(self):
        start = self.pos
        self.emit("LBRACKET", "[", start, start + 1)
        self.pos += 1
        self.lex_script(opening=self.span(start, start + 1))
        self.emit("RBRACKET", "]", self.pos, self.pos + 1)
        self.pos += 1


def tokenize_script(source: str, source_name: str = "<input>", base_offset: int = 0) -> list[Token]:
    return ScriptLexer(source, source_name, base_offset).tokenize()

Overwriting minitcl/script_lexer.py


### Parser descendente recursivo completo

Se conservan `parse`, `parse_script` y `parse_command` del esqueleto y se completan los métodos pendientes. Cada no terminal de la sección 4 es un método; cada decisión mira un solo token.

In [8]:
%%writefile minitcl/script_parser.py
"""Parser descendente recursivo de scripts MiniTcl (secciones 4, 10 y 11).

Gramática (EBNF) sobre los tokens de ``script_lexer``::

    program      = script, EOF ;
    script       = { WS | SEP }, [ command, { command_gap, command } ], { WS | SEP } ;
    command_gap  = { WS }, SEP, { WS | SEP } ;
    command      = word, { WS, word } ;
    word         = BRACED | quoted_word | bare_word ;
    quoted_word  = QOPEN, { part }, QCLOSE ;
    bare_word    = part, { part } ;
    part         = TEXT | VAR | substitution ;
    substitution = LBRACKET, script, RBRACKET ;

Cada no terminal es un método. Las decisiones se toman mirando un único
token (predicción LL(1) sobre tokens):

* FIRST(word) = {BRACED, QOPEN, TEXT, VAR, LBRACKET}
* FIRST(part) = {TEXT, VAR, LBRACKET}
* tras QOPEN, un QCLOSE inmediato da una palabra vacía;
* tras WS en un comando, se lee otra palabra solo si el token está en
  FIRST(word); si no, el espacio era final;
* tras LBRACKET, un RBRACKET inmediato da un script vacío.
"""
from __future__ import annotations

from .model import (Command, CommandSubstitution, Literal, ParseError, Part, Script,
                    Span, TokenCursor, Variable, Word, cover, normalize_newlines)
from .script_lexer import ScriptLexer


class ScriptParser(TokenCursor):
    WORD_START = {"TEXT", "VAR", "BRACED", "QOPEN", "LBRACKET"}
    PART_START = {"TEXT", "VAR", "LBRACKET"}

    def parse(self) -> Script:
        result = self.parse_script(stop="EOF")
        self.expect("EOF")
        return result

    def parse_script(self, stop="EOF") -> Script:
        start = self.current.span
        commands = []
        self.skip("WS", "SEP")
        while self.current.kind != stop:
            if self.current.kind == "EOF":
                raise ParseError(f"Falta {stop}", self.current.span,
                                 expected=stop, found="EOF")
            if self.current.kind == "RBRACKET":
                raise self.misplaced_close()
            commands.append(self.parse_command())
            if self.current.kind == stop:
                break
            if self.current.kind == "RBRACKET":
                raise self.misplaced_close()
            self.expect("SEP")
            self.skip("WS", "SEP")
        end = commands[-1].span if commands else self.current.span
        if not commands:
            # Script vacío: span de longitud cero en la posición actual.
            start = Span(self.current.span.start, self.current.span.start,
                         self.current.span.source)
            end = start
        return Script(commands=tuple(commands), span=cover(start, end))

    def misplaced_close(self) -> ParseError:
        return ParseError("Cierre ']' fuera de una sustitución de comando",
                          self.current.span, expected="palabra, separador o EOF",
                          found="RBRACKET")

    def parse_command(self) -> Command:
        words = [self.parse_word()]
        while self.accept("WS"):
            if self.current.kind not in self.WORD_START:
                break      # espacio final: no empieza otra palabra
            words.append(self.parse_word())
        return Command(words=tuple(words), span=cover(words[0].span, words[-1].span))

    # word = BRACED | quoted_word | bare_word
    def parse_word(self) -> Word:
        kind = self.current.kind
        if kind == "BRACED":
            return self.parse_braced_word()
        if kind == "QOPEN":
            return self.parse_quoted_word()
        if kind in self.PART_START:
            return self.parse_bare_word()
        raise ParseError(f"Se esperaba una palabra; se encontró {kind}",
                         self.current.span, expected="palabra", found=kind)

    # bare_word = part, { part }
    def parse_bare_word(self) -> Word:
        parts = [self.parse_part()]
        while self.current.kind in self.PART_START:
            parts.append(self.parse_part())
        return Word(parts=tuple(parts), mode="bare",
                    span=cover(parts[0].span, parts[-1].span))

    # quoted_word = QOPEN, { part }, QCLOSE
    def parse_quoted_word(self) -> Word:
        opening = self.expect("QOPEN")
        parts = []
        while self.current.kind in self.PART_START:
            parts.append(self.parse_part())
        if self.current.kind != "QCLOSE":
            raise ParseError("Faltan comillas de cierre", self.current.span,
                             expected="QCLOSE", found=self.current.kind,
                             opening=opening.span)
        closing = self.expect("QCLOSE")
        if self.current.kind not in ("WS", "SEP", "RBRACKET", "EOF"):
            raise ParseError("Texto después de cerrar una palabra entre comillas",
                             self.current.span, expected="WS, SEP, ] o EOF",
                             found=self.current.kind)
        return Word(parts=tuple(parts), mode="quoted",
                    span=cover(opening.span, closing.span))

    # BRACED produce un único Literal con el contenido sin interpretar
    def parse_braced_word(self) -> Word:
        token = self.expect("BRACED")
        inner = Span(token.span.start + 1, token.span.end - 1, token.span.source)
        literal = Literal(text=token.value, span=inner)
        if self.current.kind not in ("WS", "SEP", "RBRACKET", "EOF"):
            raise ParseError("Texto después de cerrar una palabra entre llaves",
                             self.current.span, expected="WS, SEP, ] o EOF",
                             found=self.current.kind)
        return Word(parts=(literal,), mode="braced", span=token.span)

    # part = TEXT | VAR | substitution
    def parse_part(self) -> Part:
        kind = self.current.kind
        if kind == "TEXT":
            token = self.expect("TEXT")
            return Literal(text=token.value, span=token.span)
        if kind == "VAR":
            token = self.expect("VAR")
            return Variable(name=token.value, index=token.index, span=token.span)
        if kind == "LBRACKET":
            return self.parse_substitution()
        raise ParseError(f"Se esperaba TEXT, VAR o '['; se encontró {kind}",
                         self.current.span, expected="parte de palabra", found=kind)

    # substitution = LBRACKET, script, RBRACKET
    def parse_substitution(self) -> CommandSubstitution:
        opening = self.expect("LBRACKET")
        script = self.parse_script(stop="RBRACKET")
        if self.current.kind != "RBRACKET":
            raise ParseError("Falta ']' que cierre la sustitución", self.current.span,
                             expected="RBRACKET", found=self.current.kind,
                             opening=opening.span)
        closing = self.expect("RBRACKET")
        return CommandSubstitution(script=script, span=cover(opening.span, closing.span))


def parse_minitcl(source: str, source_name: str = "<input>", *, base_offset: int = 0,
                  full_source: str | None = None) -> Script:
    """Fachada: lexer manual + parser descendente recursivo.

    ``base_offset`` permite reanalizar el interior de una palabra ``BRACED``
    conservando los offsets de la fuente completa (``full_source``), que se
    usa para calcular línea y columna de los diagnósticos.
    """
    text = normalize_newlines(source)
    try:
        tokens = ScriptLexer(text, source_name, base_offset).tokenize()
        return ScriptParser(tokens).parse()
    except ParseError as err:
        if full_source is not None:
            err.attach_source(normalize_newlines(full_source))
        elif base_offset == 0:
            err.attach_source(text)
        raise


def read_substitution(text: str, opening_index: int, base_offset: int,
                      source_name: str) -> tuple[Script, int]:
    """Callback para ``ExprLexer``: lee ``[ ... ]`` balanceado desde ``opening_index``.

    Devuelve el ``Script`` interior y el índice local inmediatamente
    posterior al ``]``. Respeta llaves, comillas y anidamiento porque usa el
    mismo lexer contextual que los scripts.
    """
    lexer = ScriptLexer(text, source_name, base_offset)
    tokens = lexer.tokenize_substitution(opening_index)
    parser = ScriptParser(tokens)
    node = parser.parse_substitution()
    parser.expect("EOF")
    return node.script, lexer.pos

Overwriting minitcl/script_parser.py


In [9]:
from minitcl.script_lexer import ScriptLexer, tokenize_script
from minitcl.script_parser import ScriptParser, parse_minitcl, read_substitution

for tok in tokenize_script('puts "Hola $nombre: [expr {2 + 3}]"'):
    print(f"{tok.kind:9} {tok.value!r:12} {tok.span.start:>2}-{tok.span.end:<2} {tok.index or ''}")

TEXT      'puts'        0-4  
WS        ' '           4-5  
QOPEN     '"'           5-6  
TEXT      'Hola '       6-11 
VAR       'nombre'     11-18 
TEXT      ': '         18-20 
LBRACKET  '['          20-21 
TEXT      'expr'       21-25 
WS        ' '          25-26 
BRACED    '2 + 3'      26-33 
RBRACKET  ']'          33-34 
QCLOSE    '"'          34-35 
EOF       ''           35-35 


In [10]:
# Tokens MANUALES: esta celda no sustituye el trabajo del lexer.
# Fuente equivalente: set x 10; puts hola
starter_tokens = [
    Token("TEXT", "set", Span(0, 3)), Token("WS", " ", Span(3, 4)),
    Token("TEXT", "x", Span(4, 5)), Token("WS", " ", Span(5, 6)),
    Token("TEXT", "10", Span(6, 8)), Token("SEP", ";", Span(8, 9)),
    Token("WS", " ", Span(9, 10)), Token("TEXT", "puts", Span(10, 14)),
    Token("WS", " ", Span(14, 15)), Token("TEXT", "hola", Span(15, 19)),
    Token("EOF", "", Span(19, 19)),
]
starter_ast = ScriptParser(starter_tokens).parse()
assert len(starter_ast.commands) == 2
assert starter_ast.commands[0].words[2].parts[0].text == "10"
show_ast(starter_ast)

Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'set'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'x'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                '10'
          mode:
            'bare'
    Command
      words:
        Word
          parts:
            Literal
              text:
                'puts'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'hola'
          mode:
            'bare'


In [11]:
# El mismo programa del arranque, ahora con tokens producidos por el lexer real.
real_ast = parse_minitcl("set x 10; puts hola")
assert real_ast == starter_ast
show_ast(parse_minitcl("set ruta prefijo_${nombre}_fin"))

Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'set'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'ruta'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'prefijo_'
            Variable
              name:
                'nombre'
              index:
                None
            Literal
              text:
                '_fin'
          mode:
            'bare'


## 11. Contratos de los métodos pendientes

| Método | Entrada / decisión | Salida y obligación |
|---|---|---|
| `parse_word` | Mirar primer token | Elegir `BRACED`, `QOPEN` o palabra no agrupada |
| `parse_bare_word` | `TEXT`, `VAR` o `LBRACKET` | Acumular partes hasta que deje de comenzar una parte; no consumir separador |
| `parse_quoted_word` | `QOPEN` | Admitir cero partes; exigir `QCLOSE`; rechazar texto adyacente tras el cierre |
| `parse_braced_word` | `BRACED` | Un `Word(mode='braced')` con un solo `Literal`; no interpretar su contenido |
| `parse_part` | Primera clase de fragmento | Producir `Literal`, `Variable` o `CommandSubstitution` |
| `parse_substitution` | `LBRACKET` | Llamar a `parse_script(stop='RBRACKET')`, consumir cierre y construir nodo |
| `parse_command` | Inicio de palabra | Acumular palabras separadas por `WS`; no confundir partes contiguas con argumentos |

### Puntos de predicción que debe justificar

- `FIRST(word) = {BRACED, QOPEN, TEXT, VAR, LBRACKET}`.
- `FIRST(part) = {TEXT, VAR, LBRACKET}`.
- Después de `QOPEN`, `QCLOSE` permite una palabra vacía.
- Después de `WS` en un comando, mirar si empieza otra palabra; de lo contrario el espacio es final.
- Después de `LBRACKET`, un `RBRACKET` inmediato admite un script vacío.

**Actividad escrita:** explique por qué el modo del lexer forma parte del contrato. No declare que toda la sintaxis de Tcl es una gramática LL(1) sobre caracteres sin mostrar cómo se tokenizan los contextos.

### Respuesta: justificación de los puntos de predicción

La gramática de la sección 4 es LL(1) **sobre la secuencia de tokens**; cada decisión del parser usa solo `self.current.kind`.

| Decisión | Conjuntos | Por qué basta un token |
|---|---|---|
| `parse_word`: elegir alternativa | FIRST(`BRACED`) = {BRACED}, FIRST(`quoted_word`) = {QOPEN}, FIRST(`bare_word`) = FIRST(`part`) = {TEXT, VAR, LBRACKET} | Los tres conjuntos son disjuntos. Cualquier otro token (`RBRACKET`, `SEP`, …) es error. |
| `parse_bare_word`: repetir `part` | FIRST(`part`) = {TEXT, VAR, LBRACKET}; FOLLOW(`bare_word`) = {WS, SEP, RBRACKET, EOF} | FIRST ∩ FOLLOW = ∅: se sigue mientras empiece una parte. El lexer nunca deja dos partes separadas por espacio sin un `WS`, por lo que las partes contiguas forman una sola palabra. |
| `parse_quoted_word`: `{ part }` hasta `QCLOSE` | QCLOSE ∉ FIRST(`part`) | Un `QCLOSE` inmediato da la palabra vacía `""` (cero partes). |
| `parse_command`: ¿otra palabra tras `WS`? | Tras consumir `WS`: FIRST(`word`) frente a {SEP, RBRACKET, EOF} | El `WS` puede pertenecer a `{ WS, word }` o ser espacio final del script. La convención operativa consume `WS` y mira el siguiente token: si está en FIRST(`word`) se lee otra palabra; si no, el espacio era final. No hay retroceso. |
| `parse_script`: ¿otro comando? | Fin de contexto `stop` ∈ {EOF, RBRACKET}; tras un comando se exige `SEP` | `SEP`, `WS` se saltan con `skip`; un `RBRACKET` en el nivel exterior se rechaza con un diagnóstico específico. |
| `parse_substitution`: script vacío | RBRACKET ∉ FIRST(`command`) | `[]` produce `Script(commands=())`; el método que abrió `[` consume el `]`. |

`parse_substitution` llama a `parse_script`, que puede volver a encontrar `LBRACKET`: es recursión válida porque antes de la llamada ya se consumió un token (no hay recursividad izquierda).

### Actividad escrita: por qué el modo del lexer forma parte del contrato

La sintaxis de Tcl **no** es LL(1) sobre caracteres. Un mismo carácter significa cosas distintas según el contexto:

- `;` y el salto de línea son `SEP` en modo script, pero texto dentro de comillas y parte del valor dentro de llaves.
- El espacio es `WS` (separa palabras) en modo script y texto dentro de comillas o llaves.
- `{` agrupa solo al comienzo de una palabra; dentro de una palabra no agrupada es un error; dentro de llaves solo cambia la profundidad.
- `]` cierra una sustitución en modo script anidado, es texto dentro de llaves y de comillas exteriores, y es un error en el nivel exterior.
- `#` es comentario solo donde empieza un comando; en `puts a#b` o `puts #dato` es texto.
- `$` y `[` abren sustituciones fuera de llaves (incluso dentro de comillas), pero no dentro de llaves.

El balanceo de llaves y de corchetes anidados no es regular: el lexer necesita un contador o una pila (aquí, la pila de llamadas). Es el lexer el que resuelve estos contextos y entrega tokens (`BRACED`, `QOPEN`, `TEXT`, `WS`, `SEP`, …) que ya los codifican; solo **sobre esos tokens** la gramática estructural es LL(1). Por eso el modo del lexer forma parte del contrato entre lexer y parser: si el lexer descartara los espacios o no distinguiera modos, el parser no podría separar palabras ni decidir con un token.

## 12. Esqueleto mínimo de expresiones con SLY

Se entrega únicamente el caso de una expresión formada por un número. `ExprLexer` hereda de `sly.Lexer` y `ExprParser` de `sly.Parser`; no heredan de `TokenCursor`.

**Actividad:** incorporar el repertorio de tokens de la sección 5, definir todas las producciones con `@_()` y devolver los nodos del AST. El lexer de scripts y su parser descendente recursivo siguen siendo manuales. No reutilizar los tokens del cursor manual como si fueran tokens SLY: tienen contratos distintos.

La celda es opcional cuando SLY no está instalado. No se simula la biblioteca con clases sustitutas. El arranque solo acepta un número, por lo que las pruebas completas siguen desactivadas hasta terminar las reglas.

In [12]:
!pip install -q sly

### Implementación con SLY

`ExprLexer` hereda de `sly.Lexer` y `ExprParser` de `sly.Parser`. Los tokens con contenido llevan `(dato, Span)` con posiciones absolutas. `[` delega en el callback `read_substitution` del parser manual, que devuelve el `Script` y el índice posterior al `]`.

In [13]:
%%writefile minitcl/expr_lexer.py
"""Lexer de expresiones de ``expr`` con ``sly.Lexer`` (sección 5).

Los patrones largos se definen antes que los cortos (``**`` antes de
``*``, ``<=`` antes de ``<``, ``==``/``!=`` antes de ``!``). ``eq``, ``ne``
y los booleanos se reconocen como palabras completas. ``[`` delega la
lectura balanceada al analizador manual mediante el callback
``read_substitution`` y produce un único token ``CMD_SUBST`` cuyo valor es
un ``Script``.

Los tokens con contenido (``NUMBER``, ``BOOL``, ``VAR``, ``STRING``,
``CMD_SUBST``) llevan como ``value`` un par ``(dato, Span)`` con el span
absoluto, ya desplazado por ``base_offset``.
"""
from __future__ import annotations

from sly import Lexer

from .model import ParseError, Span, Variable
from .script_lexer import decode_escape, scan_variable

BOOL_WORDS = {"true": True, "false": False, "yes": True, "no": False,
              "on": True, "off": False}


class ExprLexer(Lexer):
    tokens = {OR, AND, EQ, NE, STREQ, STRNE, LE, GE, LT, GT,          # noqa: F821
              POWER, PLUS, MINUS, TIMES, DIVIDE, MOD, NOT,              # noqa: F821
              LPAREN, RPAREN, NUMBER, BOOL, VAR, STRING, CMD_SUBST}     # noqa: F821
    ignore = " \t\r"

    # Operadores: largos antes que cortos.
    OR = r"\|\|"
    AND = r"&&"
    EQ = r"=="
    NE = r"!="
    LE = r"<="
    GE = r">="
    LT = r"<"
    GT = r">"
    POWER = r"\*\*"
    TIMES = r"\*"
    PLUS = r"\+"
    MINUS = r"-"
    DIVIDE = r"/"
    MOD = r"%"
    NOT = r"!"
    LPAREN = r"\("
    RPAREN = r"\)"

    def __init__(self, base_offset=0, source_name="<expr>", read_substitution=None):
        self.base_offset = base_offset
        self.source_name = source_name
        self.read_substitution = read_substitution

    def _span(self, start, end):
        return Span(self.base_offset + start, self.base_offset + end, self.source_name)

    @_(r"(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?")
    def NUMBER(self, t):
        raw = t.value
        value = float(raw) if any(c in raw for c in ".eE") else int(raw)
        t.value = (value, self._span(t.index, t.end))
        return t

    # Palabras completas: eq, ne y booleanos. Cualquier otra es error.
    @_(r"[A-Za-z_][A-Za-z0-9_]*")
    def BOOL(self, t):
        word = t.value
        if word == "eq":
            t.type = "STREQ"
        elif word == "ne":
            t.type = "STRNE"
        elif word in BOOL_WORDS:
            t.value = (BOOL_WORDS[word], self._span(t.index, t.end))
        else:
            raise ParseError(f"Palabra desnuda '{word}' no admitida en expr "
                             "(sin funciones ni cadenas sin comillas)",
                             self._span(t.index, t.end), expected="operando",
                             found=word, stage="expr")
        return t

    @_(r"\$")
    def VAR(self, t):
        name, index, end = scan_variable(self.text, t.index, self.base_offset,
                                         self.source_name, stage="expr")
        span = self._span(t.index, end)
        t.value = (Variable(name=name, index=index, span=span), span)
        self.index = t.end = end
        return t

    @_(r'"')
    def STRING(self, t):
        text = self.text
        pos = t.index + 1
        chars = []
        while pos < len(text):
            c = text[pos]
            if c == '"':
                end = pos + 1
                t.value = ("".join(chars), self._span(t.index, end))
                self.lineno += text.count("\n", t.index, end)
                self.index = t.end = end
                return t
            if c == "\\":
                ch, pos = decode_escape(text, pos, self.base_offset, self.source_name, "expr")
                chars.append(ch)
                continue
            if c in "$[":
                raise ParseError(f"'{c}' no escapado dentro de una cadena de expr "
                                 "(sin interpolación en este perfil)",
                                 self._span(pos, pos + 1), expected="carácter literal",
                                 found=c, stage="expr")
            chars.append(c)
            pos += 1
        raise ParseError('Cadena sin comillas de cierre en expr', self._span(len(text), len(text)),
                         expected='"', found="EOF", opening=self._span(t.index, t.index + 1),
                         stage="expr")

    @_(r"\[")
    def CMD_SUBST(self, t):
        if self.read_substitution is None:
            raise ParseError("Sustitución de comando sin callback read_substitution",
                             self._span(t.index, t.index + 1), stage="expr")
        script, end = self.read_substitution(self.text, t.index, self.base_offset,
                                             self.source_name)
        t.value = (script, self._span(t.index, end))
        self.lineno += self.text.count("\n", t.index, end)
        self.index = t.end = end
        return t

    @_(r"\n+")
    def ignore_newline(self, t):
        self.lineno += len(t.value)

    def error(self, t):
        position = self.base_offset + t.index
        raise ParseError(f"Carácter inválido en expr: {t.value[0]!r}",
                         Span(position, position + 1, self.source_name),
                         expected="operador u operando", found=t.value[0], stage="expr")

Overwriting minitcl/expr_lexer.py


In [14]:
%%writefile minitcl/expr_parser.py
"""Parser LALR(1) de expresiones con ``sly.Parser`` (sección 5).

La gramática es por niveles; los binarios usan recursividad izquierda
(asociación a la izquierda) y la potencia ``power : primary POWER unary``
asocia a la derecha y deja a los unarios por encima de ``**`` como
operando izquierdo: ``-2 ** 2 = -(2 ** 2)`` y ``2 ** -3 = 2 ** (-3)``.
No se usa tabla ``precedence``: los niveles ya la definen, y la tabla
generada no tiene conflictos.

Las acciones solo construyen nodos del AST; nunca evalúan.
"""
from __future__ import annotations

from sly import Parser

from .expr_lexer import ExprLexer
from .model import (Binary, Constant, Expr, ParseError, Span, SubstitutionExpr, Unary,
                    VariableExpr, cover, normalize_newlines)
from .script_parser import read_substitution


class ExprParser(Parser):
    tokens = ExprLexer.tokens
    start = "expression"

    def __init__(self, text="", base_offset=0, source_name="<expr>"):
        self.text = text
        self.base_offset = base_offset
        self.source_name = source_name

    def _binary(self, op, left, right):
        return Binary(operator=op, left=left, right=right, span=cover(left.span, right.span))

    def _unary(self, op, index, operand):
        start = Span(self.base_offset + index, self.base_offset + index + 1, self.source_name)
        return Unary(operator=op, operand=operand, span=cover(start, operand.span))

    # expression = logical_or
    @_("logical_or")
    def expression(self, p):
        return p.logical_or

    # logical_or = logical_or "||" logical_and | logical_and
    @_("logical_or OR logical_and")
    def logical_or(self, p):
        return self._binary("||", p.logical_or, p.logical_and)

    @_("logical_and")
    def logical_or(self, p):  # noqa: F811
        return p.logical_and

    # logical_and = logical_and "&&" equality | equality
    @_("logical_and AND equality")
    def logical_and(self, p):
        return self._binary("&&", p.logical_and, p.equality)

    @_("equality")
    def logical_and(self, p):  # noqa: F811
        return p.equality

    # equality = equality ("==" | "!=" | "eq" | "ne") relational | relational
    @_("equality EQ relational", "equality NE relational",
       "equality STREQ relational", "equality STRNE relational")
    def equality(self, p):
        return self._binary(p[1], p.equality, p.relational)

    @_("relational")
    def equality(self, p):  # noqa: F811
        return p.relational

    # relational = relational ("<" | "<=" | ">" | ">=") additive | additive
    @_("relational LT additive", "relational LE additive",
       "relational GT additive", "relational GE additive")
    def relational(self, p):
        return self._binary(p[1], p.relational, p.additive)

    @_("additive")
    def relational(self, p):  # noqa: F811
        return p.additive

    # additive = additive ("+" | "-") multiplicative | multiplicative
    @_("additive PLUS multiplicative", "additive MINUS multiplicative")
    def additive(self, p):
        return self._binary(p[1], p.additive, p.multiplicative)

    @_("multiplicative")
    def additive(self, p):  # noqa: F811
        return p.multiplicative

    # multiplicative = multiplicative ("*" | "/" | "%") unary | unary
    @_("multiplicative TIMES unary", "multiplicative DIVIDE unary",
       "multiplicative MOD unary")
    def multiplicative(self, p):
        return self._binary(p[1], p.multiplicative, p.unary)

    @_("unary")
    def multiplicative(self, p):  # noqa: F811
        return p.unary

    # unary = ("+" | "-" | "!") unary | power
    @_("PLUS unary", "MINUS unary", "NOT unary")
    def unary(self, p):
        return self._unary(p[0], p.index, p.unary)

    @_("power")
    def unary(self, p):  # noqa: F811
        return p.power

    # power = primary "**" unary | primary
    @_("primary POWER unary")
    def power(self, p):
        return self._binary("**", p.primary, p.unary)

    @_("primary")
    def power(self, p):  # noqa: F811
        return p.primary

    # primary = NUMBER | BOOL | VAR | STRING | "(" expression ")" | CMD_SUBST
    @_("NUMBER", "BOOL", "STRING")
    def primary(self, p):
        value, span = p[0]
        return Constant(value=value, span=span)

    @_("VAR")
    def primary(self, p):  # noqa: F811
        variable, span = p.VAR
        return VariableExpr(variable=variable, span=span)

    @_("CMD_SUBST")
    def primary(self, p):  # noqa: F811
        script, span = p.CMD_SUBST
        return SubstitutionExpr(script=script, span=span)

    @_("LPAREN expression RPAREN")
    def primary(self, p):  # noqa: F811
        return p.expression        # los paréntesis no generan nodo propio

    def error(self, token):
        if token is None:
            position = self.base_offset + len(self.text)
            raise ParseError("Fin inesperado de expresión",
                             Span(position, position, self.source_name),
                             expected="operando", found="fin", stage="expr")
        position = self.base_offset + token.index
        end = getattr(token, "end", token.index + 1) or token.index + 1
        raise ParseError(f"Token inesperado en expr: {token.type}",
                         Span(position, self.base_offset + end, self.source_name),
                         expected="operador u operando", found=token.type, stage="expr")


def parse_minitcl_expr(source: str, *, base_offset=0, source_name="<expr>",
                       full_source: str | None = None) -> Expr:
    """Fachada única para ``expr`` y las condiciones de ``if``/``while``/``for``.

    Recibe solo el contenido literal (sin ``expr`` ni las llaves). Crea
    instancias nuevas de lexer y parser en cada llamada para que una
    sustitución anidada no sobrescriba un análisis en curso.
    """
    text = normalize_newlines(source)
    lexer = ExprLexer(base_offset=base_offset, source_name=source_name,
                      read_substitution=read_substitution)
    parser = ExprParser(text=text, base_offset=base_offset, source_name=source_name)
    try:
        result = parser.parse(lexer.tokenize(text))
        if result is None:   # SLY devuelve None para una entrada vacía
            raise ParseError("Expresión vacía", Span(base_offset, base_offset, source_name),
                             expected="expresión", found="fin", stage="expr")
        return result
    except ParseError as err:
        if full_source is not None:
            err.attach_source(normalize_newlines(full_source))
        elif base_offset == 0:
            err.attach_source(text)
        raise

Overwriting minitcl/expr_parser.py


In [15]:
from sly import Lexer, Parser
from minitcl.expr_lexer import ExprLexer
from minitcl.expr_parser import ExprParser, parse_minitcl_expr

assert issubclass(ExprLexer, Lexer) and issubclass(ExprParser, Parser)
table = ExprParser._lrtable
print(f"Tabla LALR(1): {len(table.lr_action)} estados, "
      f"{len(table.sr_conflicts)} conflictos shift/reduce, {len(table.rr_conflicts)} reduce/reduce")
assert not table.sr_conflicts and not table.rr_conflicts

number_ast = parse_minitcl_expr('42')
assert isinstance(number_ast, Constant) and number_ast.value == 42
print([ (t.type, t.value if t.type not in ('VAR', 'CMD_SUBST') else '...')
        for t in ExprLexer(read_substitution=read_substitution).tokenize('$x + [doble 2] ** -1.5')])

Tabla LALR(1): 57 estados, 0 conflictos shift/reduce, 0 reduce/reduce
[('VAR', '...'), ('PLUS', '+'), ('CMD_SUBST', '...'), ('POWER', '**'), ('MINUS', '-'), ('NUMBER', (1.5, Span(start=19, end=22, source='<expr>')))]


### Cómo traducir las producciones a reglas de SLY

Traduzca cada alternativa de la sección 5 a una producción decorada con `@_()`. En una regla binaria como `additive PLUS multiplicative`, la acción debe construir `Binary` con los dos AST y un span que cubra la operación. En una regla de paso, como `additive -> multiplicative`, devuelva el AST ya construido.

**Actividad:** explicar cómo la recursividad izquierda de `additive` produce la asociación de `10 - 3 - 2`, y cómo `power -> primary POWER unary` produce la asociación de `2 ** 3 ** 2`. Esta recursividad izquierda es válida en SLY; no debe introducirse en el parser descendente recursivo de scripts.

No se entrega una acción por cada operador: el estudiante debe completarlas. Dibuje antes los AST de `8 / 2 * 3`, `2 ** 3 ** 2`, `-(4 + 1)` y `$a(3) + [doble 2]`, sin evaluarlos.

### Respuesta: asociatividad a partir de la forma de las producciones

**`10 - 3 - 2`.** En `additive : additive MINUS multiplicative` el operando izquierdo es otro `additive` y el derecho un `multiplicative`, que no puede contener un `-` sin paréntesis. El LALR(1) desplaza `10 - 3`; al ver el segundo `-` no puede seguir desplazando dentro del `multiplicative`, así que reduce `10 - 3` a `additive` y lo usa como operando izquierdo del siguiente `-`. Resultado: `Binary('-', Binary('-', 10, 3), 2)`. SLY admite recursividad izquierda porque es ascendente (usa una pila de estados); un método descendente recursivo `parse_additive()` que se llamara a sí mismo primero entraría en recursión infinita, por eso el parser de scripts solo usa repeticiones.

**`2 ** 3 ** 2`.** En `power : primary POWER unary` el operando izquierdo es un `primary` (no puede contener `**`) y el derecho un `unary`, que deriva otra vez `power`. Tras `2 **` el parser debe completar un `unary` completo, que absorbe `3 ** 2`. Resultado: `Binary('**', 2, Binary('**', 3, 2))`: asociación derecha.

**Unarios y potencia.** `unary : MINUS unary | power` coloca a los unarios por encima de `power`: en `-2 ** 2` el `-` toma como operando todo el `power` → `Unary('-', Binary('**', 2, 2))`. En `2 ** -3`, el operando derecho de `**` es un `unary`, así que admite el signo → `Binary('**', 2, Unary('-', 3))`.

No se necesita tabla `precedence`: los niveles la codifican y la tabla generada tiene **0 conflictos**.

Los AST pedidos (sin evaluar) se obtienen en la celda siguiente:

In [16]:
def expr_tree(node, indent=0):
    pad = "  " * indent
    if isinstance(node, Constant):
        print(f"{pad}Constant({node.value!r})")
    elif isinstance(node, VariableExpr):
        v = node.variable
        print(f"{pad}VariableExpr({v.name}{'(' + v.index + ')' if v.index else ''})")
    elif isinstance(node, SubstitutionExpr):
        print(f"{pad}SubstitutionExpr(Script con {len(node.script.commands)} comando(s))")
    elif isinstance(node, Unary):
        print(f"{pad}Unary({node.operator!r})")
        expr_tree(node.operand, indent + 1)
    else:
        print(f"{pad}Binary({node.operator!r})")
        expr_tree(node.left, indent + 1)
        expr_tree(node.right, indent + 1)

for src in ["8 / 2 * 3", "2 ** 3 ** 2", "-(4 + 1)", "$a(3) + [doble 2]"]:
    print(f"--- {src}")
    expr_tree(parse_minitcl_expr(src))

--- 8 / 2 * 3
Binary('*')
  Binary('/')
    Constant(8)
    Constant(2)
  Constant(3)
--- 2 ** 3 ** 2
Binary('**')
  Constant(2)
  Binary('**')
    Constant(3)
    Constant(2)
--- -(4 + 1)
Unary('-')
  Binary('+')
    Constant(4)
    Constant(1)
--- $a(3) + [doble 2]
Binary('+')
  VariableExpr(a(3))
  SubstitutionExpr(Script con 1 comando(s))


## 13. Análisis de cuerpos y condiciones, sin ejecutarlos

El AST estructural basta para guardar el programa, pero una entrega completa del análisis sintáctico debe demostrar cómo reutilizar ambos parsers en los contextos estáticos.

| Comando | Argumentos que se vuelven a analizar |
|---|---|
| `expr` | Expresión única |
| `if` / `elseif` | Cada condición con `ExprParser` de SLY; cada cuerpo con `ScriptParser` manual |
| `while` | Condición como expresión; cuerpo como script |
| `for` | Inicio, paso y cuerpo como scripts; condición como expresión |
| `foreach` | Cuerpo como script; lista sigue siendo una palabra ordinaria |
| `proc` | Parámetros como lista de nombres; cuerpo como script |
| `catch` | Cuerpo como script |

**Actividad:** escribir `analyze_command(command)` y una estructura de resultados que vincule cada argumento con su AST adicional. Mantener el AST estructural original. En la pasada estática se pueden analizar cuerpos no ejecutados; un error de sintaxis dentro de un cuerpo estático debe reportarse aunque no se tome esa rama.

Al reanalizar un `BRACED`, la primera posición interior es `word.span.start + 1`; propague ese offset y el nombre de la fuente. No desplace las posiciones calculándolas sobre texto interpolado: el perfil exige cuerpos literales. No interprete como scripts los bloques de `set`, `puts`, `list` o los datos de `dict`.

Los corchetes ya producen un `Script` durante el análisis estructural. Visite también esos scripts al validar. Una llamada a un procedimiento definido más adelante no es por sí sola un error sintáctico.

### Fachada común para expresiones

`parse_minitcl_expr(source, *, base_offset=0, source_name='<expr>') -> Expr` integra `ExprLexer` y `ExprParser` de SLY y consume toda la entrada. No recibe `expr` ni las llaves exteriores, solo su contenido literal. El analizador de comandos la llama para `expr`, `if`, `elseif`, `while` y `for`; no debe existir un segundo parser manual para condiciones.

En `set r [expr {$x + 3}]`, el lexer manual conserva el argumento de `expr` como `BRACED`. La pasada de análisis entrega `$x + 3` a SLY y asocia su `Binary` al argumento original. En `expr {$x + [doble 2]}`, el lexer SLY devuelve `VAR`, `PLUS`, `CMD_SUBST`; ese último token contiene el `Script` producido por el analizador manual. Ningún paso consulta el valor de `x` ni ejecuta `doble`.


### Implementación: lector de listas y analizador de comandos

`analyze_script` recorre el AST estructural sin modificarlo y produce `AnalysisResult` con tres mapas: `bodies` (palabra → `Script`), `exprs` (palabra → `Expr`) y `params` (palabra → nombres). Los cuerpos se reanalizan con `parse_minitcl(..., base_offset=literal.span.start)` y las condiciones con la **misma** fachada `parse_minitcl_expr` que usa `expr`. Un `Context(in_proc, in_loop)` permite validar `return`, `break` y `continue`.

In [17]:
%%writefile minitcl/list_reader.py
"""Lector y formateador de listas MiniTcl (sección 7).

El contenido de una lista no es un script: ``$x``, ``[comando]`` y ``;``
son datos. El salto de línea separa elementos igual que un espacio.

    list_value = { LIST_WS }, [ element, { LIST_WS, element } ], { LIST_WS } ;
    element    = LIST_BRACED | LIST_QUOTED | LIST_BARE ;

Contrato: ``parse_list(format_list(values)) == values``.
"""
from __future__ import annotations

from .model import ParseError, Span
from .script_lexer import decode_escape

LIST_WS = " \t\n"


def parse_list(text: str, *, base_offset: int = 0, source_name: str = "<list>") -> list[str]:
    def span(a, b):
        return Span(base_offset + a, base_offset + b, source_name)

    def err(message, a, b=None, **kw):
        return ParseError(message, span(a, b if b is not None else a + 1),
                          stage="validation", **kw)

    items: list[str] = []
    pos = 0
    n = len(text)
    while True:
        while pos < n and text[pos] in LIST_WS:
            pos += 1
        if pos >= n:
            return items
        c = text[pos]
        if c == "{":                                  # LIST_BRACED
            start, depth, i = pos, 0, pos
            while i < n:
                ch = text[i]
                if ch == "\\" and i + 1 < n:
                    i += 2
                    continue
                if ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        break
                i += 1
            else:
                raise err("Falta '}' en elemento de lista", n, n, expected="}",
                          found="EOF", opening=span(start, start + 1))
            items.append(text[start + 1:i])
            pos = i + 1
            grouped = True
        elif c == '"':                                # LIST_QUOTED
            start, i, chars = pos, pos + 1, []
            while i < n and text[i] != '"':
                if text[i] == "\\":
                    ch, i = decode_escape(text, i, base_offset, source_name, "validation")
                    chars.append(ch)
                else:
                    chars.append(text[i])
                    i += 1
            if i >= n:
                raise err("Faltan comillas de cierre en elemento de lista", n, n,
                          expected='"', found="EOF", opening=span(start, start + 1))
            items.append("".join(chars))
            pos = i + 1
            grouped = True
        else:                                         # LIST_BARE
            chars = []
            while pos < n and text[pos] not in LIST_WS:
                ch = text[pos]
                if ch == "\\":
                    ch, pos = decode_escape(text, pos, base_offset, source_name, "validation")
                    chars.append(ch)
                    continue
                if ch in '{}"':
                    raise err(f"'{ch}' no escapado en elemento de lista no agrupado", pos,
                              expected="texto o escape", found=ch)
                chars.append(ch)
                pos += 1
            items.append("".join(chars))
            grouped = False
        if grouped and pos < n and text[pos] not in LIST_WS:
            raise err("Texto después de un elemento agrupado de lista", pos,
                      expected="blanco o fin", found=text[pos])


def _brace_safe(value: str) -> bool:
    """¿Puede escribirse ``value`` entre llaves y leerse sin cambios?"""
    depth, i = 0, 0
    while i < len(value):
        ch = value[i]
        if ch == "\\":
            if i + 1 >= len(value):
                return False          # barra final escaparía la llave de cierre
            i += 2
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth < 0:
                return False
        i += 1
    return depth == 0


_BARE_ESCAPES = {"\\": "\\\\", "{": "\\{", "}": "\\}", '"': '\\"', " ": "\\ ",
                 "\t": "\\t", "\n": "\\n", "\r": "\\r"}


def format_element(value: str) -> str:
    if value == "":
        return "{}"
    if not any(ch in value for ch in ' \t\n\r{}"\\'):
        return value
    if _brace_safe(value):
        return "{" + value + "}"
    return "".join(_BARE_ESCAPES.get(ch, ch) for ch in value)


def format_list(values) -> str:
    """Serializa ``values`` sin usar ``' '.join`` ingenuo."""
    return " ".join(format_element(v) for v in values)

Overwriting minitcl/list_reader.py


In [18]:
%%writefile minitcl/command_analyzer.py
"""Validación de comandos y reanálisis de cuerpos estáticos (secciones 6 y 13).

Esta pasada se ejecuta **después** del análisis estructural. Conserva el
AST original y construye vistas adicionales:

* ``bodies``: palabra ``BRACED`` → ``Script`` (parser manual descendente).
* ``exprs``: palabra ``BRACED`` → ``Expr`` (fachada SLY ``parse_minitcl_expr``).
* ``params``: palabra de parámetros de ``proc`` → nombres.

Comprueba aridad, formas de los incorporados y el perfil estático del
curso. No ejecuta nada: no consulta variables ni llama procedimientos.
Se detiene en el primer error (``ParseError``).
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field

from .expr_parser import parse_minitcl_expr
from .list_reader import parse_list
from .model import (Command, CommandSubstitution, Expr, Literal, ParseError, Script,
                    SubstitutionExpr, Unary, Binary, Word, normalize_newlines)
from .script_parser import parse_minitcl

IDENTIFIER = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
NAME = re.compile(r"[A-Za-z_][A-Za-z0-9_]*(?:\([A-Za-z0-9_.:\-]+\))?")

BUILTINS = {
    "set", "unset", "incr", "append", "puts", "expr", "if", "while", "for",
    "foreach", "proc", "return", "break", "continue", "global", "catch", "error",
    "list", "llength", "lindex", "lappend", "lset", "dict", "string", "split", "join",
}


@dataclass
class Context:
    in_proc: bool = False
    in_loop: bool = False


@dataclass
class AnalysisResult:
    script: Script                                   # AST estructural intacto
    bodies: dict[Word, Script] = field(default_factory=dict)
    exprs: dict[Word, Expr] = field(default_factory=dict)
    params: dict[Word, tuple[str, ...]] = field(default_factory=dict)
    procedures: dict[str, Command] = field(default_factory=dict)

    def body(self, word: Word) -> Script:
        return self.bodies[word]

    def expr(self, word: Word) -> Expr:
        return self.exprs[word]


def literal_text(word: Word) -> str | None:
    """Texto de una palabra sin sustituciones; ``None`` si es calculada."""
    if all(isinstance(p, Literal) for p in word.parts):
        return "".join(p.text for p in word.parts)
    return None


class CommandAnalyzer:
    def __init__(self, result: AnalysisResult, source: str | None = None):
        self.result = result
        self.source = normalize_newlines(source) if source is not None else None

    # -- utilidades ---------------------------------------------------------
    def fail(self, message, node, *, expected=None, found=None):
        err = ParseError(message, node.span, expected=expected, found=found,
                         stage="validation")
        if self.source is not None:
            err.attach_source(self.source)
        return err

    def arity(self, cmd: Command, name: str, low: int, high: int | None, usage: str):
        count = len(cmd.words) - 1
        if count < low or (high is not None and count > high):
            raise self.fail(f"Aridad inválida en '{name}': se recibieron {count} argumentos",
                            cmd, expected=usage, found=f"{count} argumentos")

    def require_name(self, word: Word, what="nombre de variable"):
        text = literal_text(word)
        if text is None or not NAME.fullmatch(text):
            raise self.fail(f"Se esperaba {what} literal (sin '$')", word,
                            expected="identificador o a(indice)", found=text or "palabra calculada")
        return text

    def require_identifier(self, word: Word, what="identificador"):
        text = literal_text(word)
        if text is None or not IDENTIFIER.fullmatch(text):
            raise self.fail(f"Se esperaba {what}", word, expected="identificador",
                            found=text or "palabra calculada")
        return text

    def require_braced(self, word: Word, what: str):
        if word.mode != "braced":
            raise self.fail(f"Perfil estático: {what} debe escribirse entre llaves", word,
                            expected=f"{what} entre llaves", found=f"palabra {word.mode}")
        return word.parts[0]

    # -- reanálisis ---------------------------------------------------------
    def body(self, word: Word, ctx: Context) -> Script:
        literal = self.require_braced(word, "el cuerpo")
        script = parse_minitcl(literal.text, literal.span.source,
                               base_offset=literal.span.start, full_source=self.source)
        self.result.bodies[word] = script
        self.visit_script(script, ctx)
        return script

    def test(self, word: Word, ctx: Context) -> Expr:
        literal = self.require_braced(word, "la expresión")
        expr = parse_minitcl_expr(literal.text, base_offset=literal.span.start,
                                  source_name=literal.span.source, full_source=self.source)
        self.result.exprs[word] = expr
        self.visit_expr(expr, ctx)
        return expr

    def visit_expr(self, expr: Expr, ctx: Context):
        if isinstance(expr, SubstitutionExpr):
            self.visit_script(expr.script, ctx)
        elif isinstance(expr, Unary):
            self.visit_expr(expr.operand, ctx)
        elif isinstance(expr, Binary):
            self.visit_expr(expr.left, ctx)
            self.visit_expr(expr.right, ctx)

    def visit_words(self, cmd: Command, ctx: Context):
        """Visita los scripts de los corchetes ya producidos estructuralmente."""
        for word in cmd.words:
            for part in word.parts:
                if isinstance(part, CommandSubstitution):
                    self.visit_script(part.script, ctx)

    def visit_script(self, script: Script, ctx: Context):
        for cmd in script.commands:
            self.analyze(cmd, ctx)

    # -- formas de comandos --------------------------------------------------
    def analyze(self, cmd: Command, ctx: Context):
        self.visit_words(cmd, ctx)
        name = literal_text(cmd.words[0])
        if name is None:
            raise self.fail("Nombre de comando calculado: válido estructuralmente pero "
                            "fuera del perfil estático", cmd.words[0],
                            expected="nombre literal", found="palabra calculada")
        handler = getattr(self, f"cmd_{name}", None)
        if handler is not None and name in BUILTINS:
            handler(cmd, ctx)
        # Cualquier otro nombre es call_cmd: un procedimiento definido antes
        # o después; no es error sintáctico.

    def cmd_set(self, cmd, ctx):
        self.arity(cmd, "set", 1, 2, "set nombre ?valor?")
        self.require_name(cmd.words[1])

    def cmd_unset(self, cmd, ctx):
        self.arity(cmd, "unset", 1, 1, "unset nombre")
        self.require_name(cmd.words[1])

    def cmd_incr(self, cmd, ctx):
        self.arity(cmd, "incr", 1, 2, "incr nombre ?incremento?")
        self.require_name(cmd.words[1])

    def cmd_append(self, cmd, ctx):
        self.arity(cmd, "append", 2, None, "append nombre valor ?valor ...?")
        self.require_name(cmd.words[1])

    def cmd_puts(self, cmd, ctx):
        self.arity(cmd, "puts", 1, 2, "puts ?-nonewline? texto")
        if len(cmd.words) == 3 and literal_text(cmd.words[1]) != "-nonewline":
            raise self.fail("Opción desconocida de puts", cmd.words[1],
                            expected="-nonewline", found=literal_text(cmd.words[1]))

    def cmd_expr(self, cmd, ctx):
        self.arity(cmd, "expr", 1, 1, "expr {expresión}")
        self.test(cmd.words[1], ctx)

    def cmd_if(self, cmd, ctx):
        words = cmd.words
        i = 1

        def need(kind):
            if i >= len(words):
                raise self.fail(f"Forma incompleta de 'if': falta {kind}", cmd,
                                expected=kind, found="fin del comando")
            return words[i]

        self.test(need("condición"), ctx); i += 1
        if i < len(words) and literal_text(words[i]) == "then":
            i += 1
        self.body(need("cuerpo"), ctx); i += 1
        while i < len(words) and literal_text(words[i]) == "elseif":
            i += 1
            self.test(need("condición de elseif"), ctx); i += 1
            if i < len(words) and literal_text(words[i]) == "then":
                i += 1
            self.body(need("cuerpo de elseif"), ctx); i += 1
        if i < len(words) and literal_text(words[i]) == "else":
            i += 1
            self.body(need("cuerpo de else"), ctx); i += 1
        if i < len(words):
            raise self.fail("Argumento inesperado en 'if'", words[i],
                            expected="elseif, else o fin", found=literal_text(words[i]))

    def cmd_while(self, cmd, ctx):
        self.arity(cmd, "while", 2, 2, "while {test} {cuerpo}")
        self.test(cmd.words[1], ctx)
        self.body(cmd.words[2], Context(ctx.in_proc, True))

    def cmd_for(self, cmd, ctx):
        self.arity(cmd, "for", 4, 4, "for {inicio} {test} {paso} {cuerpo}")
        self.body(cmd.words[1], ctx)
        self.test(cmd.words[2], ctx)
        self.body(cmd.words[3], ctx)
        self.body(cmd.words[4], Context(ctx.in_proc, True))

    def cmd_foreach(self, cmd, ctx):
        self.arity(cmd, "foreach", 3, 3, "foreach variable lista {cuerpo}")
        self.require_identifier(cmd.words[1], "variable de foreach")
        self.body(cmd.words[3], Context(ctx.in_proc, True))

    def cmd_proc(self, cmd, ctx):
        self.arity(cmd, "proc", 3, 3, "proc nombre {parámetros} {cuerpo}")
        name = self.require_identifier(cmd.words[1], "nombre de procedimiento")
        if name in BUILTINS:
            raise self.fail(f"No se puede redefinir el incorporado '{name}'", cmd.words[1],
                            expected="nombre nuevo", found=name)
        params_word = cmd.words[2]
        literal = self.require_braced(params_word, "la lista de parámetros")
        names = parse_list(literal.text, base_offset=literal.span.start,
                           source_name=literal.span.source)
        seen = set()
        for param in names:
            if not IDENTIFIER.fullmatch(param):
                raise self.fail(f"Parámetro inválido '{param}'", params_word,
                                expected="identificador", found=param)
            if param in seen:
                raise self.fail(f"Parámetro duplicado '{param}'", params_word,
                                expected="nombres únicos", found=param)
            seen.add(param)
        self.result.params[params_word] = tuple(names)
        self.result.procedures[name] = cmd
        self.body(cmd.words[3], Context(in_proc=True, in_loop=False))

    def cmd_return(self, cmd, ctx):
        self.arity(cmd, "return", 0, 1, "return ?valor?")
        if not ctx.in_proc:
            raise self.fail("'return' solo es válido dentro de un procedimiento", cmd)

    def cmd_break(self, cmd, ctx):
        self.arity(cmd, "break", 0, 0, "break")
        if not ctx.in_loop:
            raise self.fail("'break' requiere un ciclo en el procedimiento actual", cmd)

    def cmd_continue(self, cmd, ctx):
        self.arity(cmd, "continue", 0, 0, "continue")
        if not ctx.in_loop:
            raise self.fail("'continue' requiere un ciclo en el procedimiento actual", cmd)

    def cmd_global(self, cmd, ctx):
        self.arity(cmd, "global", 1, None, "global nombre ?nombre ...?")
        for word in cmd.words[1:]:
            self.require_identifier(word, "nombre global")

    def cmd_catch(self, cmd, ctx):
        self.arity(cmd, "catch", 1, 2, "catch {cuerpo} ?variable?")
        self.body(cmd.words[1], ctx)
        if len(cmd.words) == 3:
            self.require_identifier(cmd.words[2], "variable de catch")

    def cmd_error(self, cmd, ctx):
        self.arity(cmd, "error", 1, 1, "error mensaje")

    # -- biblioteca de datos ------------------------------------------------
    def cmd_list(self, cmd, ctx):
        pass

    def cmd_llength(self, cmd, ctx):
        self.arity(cmd, "llength", 1, 1, "llength lista")

    def cmd_lindex(self, cmd, ctx):
        self.arity(cmd, "lindex", 2, 2, "lindex lista indice")

    def cmd_lappend(self, cmd, ctx):
        self.arity(cmd, "lappend", 1, None, "lappend nombre ?valor ...?")
        self.require_name(cmd.words[1])

    def cmd_lset(self, cmd, ctx):
        self.arity(cmd, "lset", 3, 3, "lset nombre indice valor")
        self.require_name(cmd.words[1])

    def cmd_dict(self, cmd, ctx):
        self.arity(cmd, "dict", 1, None, "dict subcomando ...")
        sub = literal_text(cmd.words[1])
        count = len(cmd.words) - 2
        if sub == "create":
            if count % 2:
                raise self.fail("'dict create' exige un número par de argumentos", cmd,
                                expected="pares clave valor", found=f"{count} argumentos")
        elif sub in ("get", "exists"):
            if count != 2:
                raise self.fail(f"Aridad inválida en 'dict {sub}'", cmd,
                                expected=f"dict {sub} diccionario clave", found=f"{count} argumentos")
        elif sub == "set":
            if count != 3:
                raise self.fail("Aridad inválida en 'dict set'", cmd,
                                expected="dict set nombre clave valor", found=f"{count} argumentos")
            self.require_name(cmd.words[2])
        else:
            raise self.fail("Subcomando de dict desconocido", cmd.words[1],
                            expected="create, get, exists o set", found=sub or "palabra calculada")

    def cmd_string(self, cmd, ctx):
        self.arity(cmd, "string", 1, None, "string subcomando ...")
        sub = literal_text(cmd.words[1])
        count = len(cmd.words) - 2
        expected = {"length": 1, "tolower": 1, "toupper": 1, "compare": 2}
        if sub not in expected:
            raise self.fail("Subcomando de string desconocido", cmd.words[1],
                            expected="length, tolower, toupper o compare",
                            found=sub or "palabra calculada")
        if count != expected[sub]:
            raise self.fail(f"Aridad inválida en 'string {sub}'", cmd,
                            expected=f"{expected[sub]} argumentos", found=f"{count} argumentos")

    def cmd_split(self, cmd, ctx):
        self.arity(cmd, "split", 1, 2, "split texto ?separadores?")

    def cmd_join(self, cmd, ctx):
        self.arity(cmd, "join", 1, 2, "join lista ?separador?")


def analyze_command(command: Command, *, source: str | None = None,
                    context: Context | None = None,
                    result: AnalysisResult | None = None) -> AnalysisResult:
    """Valida un comando y reanaliza sus cuerpos y condiciones estáticos."""
    if result is None:
        result = AnalysisResult(script=Script(commands=(command,), span=command.span))
    CommandAnalyzer(result, source).analyze(command, context or Context())
    return result


def analyze_script(script: Script, *, source: str | None = None) -> AnalysisResult:
    result = AnalysisResult(script=script)
    CommandAnalyzer(result, source).visit_script(script, Context())
    return result


def analyze_source(source: str, source_name: str = "<input>") -> AnalysisResult:
    """Análisis completo: estructural + validación + cuerpos y expresiones."""
    script = parse_minitcl(source, source_name)
    return analyze_script(script, source=source)

Overwriting minitcl/command_analyzer.py


In [19]:
from minitcl.list_reader import parse_list, format_list
from minitcl.command_analyzer import analyze_command, analyze_script, analyze_source, AnalysisResult

values = ["a b", "", "{x}", "}{", "$y", "c"]
assert parse_list(format_list(values)) == values
print("format_list:", format_list(values))

source = "set r [expr {$x + [doble 2]}]"
result = analyze_source(source)
for word, expr in result.exprs.items():
    print("Argumento", repr(source[word.span.start:word.span.end]), "->")
    expr_tree(expr, 1)

format_list: {a b} {} {{x}} \}\{ $y c
Argumento '{$x + [doble 2]}' ->
  Binary('+')
    VariableExpr(x)
    SubstitutionExpr(Script con 1 comando(s))


## 14. Semántica mínima para que el AST tenga un significado claro

Estas reglas son el contrato para la etapa posterior; **no se pide implementar el evaluador en este cuaderno**.

- Los valores visibles son cadenas; números, listas y diccionarios se convierten al usarlos. El intérprete puede conservar representaciones internas auxiliares.
- Las palabras ordinarias evalúan sus partes de izquierda a derecha y concatenan sus resultados. El resultado de una sustitución no se vuelve a dividir en argumentos ni se vuelve a interpretar como código.
- Una palabra `braced` entrega su literal. Una condición entre llaves se analiza una vez como expresión y se evalúa cada vez que el control la necesita; las variables no se congelan al entrar al ciclo.
- Cada llamada de procedimiento crea un ámbito local; `global` enlaza nombres globales. Un arreglo tiene la misma política de ámbito que una variable escalar. No se permite que un nombre sea simultáneamente escalar y arreglo.
- Un procedimiento devuelve el valor explícito de `return` o el del último comando. Un script vacío devuelve `""`. `set`, `incr`, `append`, `lappend`, `lset` y `dict set` devuelven el nuevo valor. `unset`, `puts`, `global` y `proc` devuelven `""`.
- `if` devuelve el resultado del cuerpo elegido o `""` si no se elige ninguno. `while`, `for` y `foreach` devuelven `""`.
- `return` solo es válido en procedimientos. `break` y `continue` requieren un ciclo en el procedimiento actual; no pueden escapar hacia un ciclo del llamador. En un `for`, `continue` en el cuerpo ejecuta el paso antes de la siguiente condición. En `foreach`, pasa al siguiente elemento.
- `catch {script} ?variable?` devuelve `0` para terminación normal y `1` para error; guarda resultado o mensaje si se da variable. En este perfil, `return`, `break` y `continue` se propagan, no se convierten a códigos adicionales. Esto es una simplificación respecto de Tcl.
- `error mensaje` interrumpe con un error. Variable inexistente, conversión imposible o aridad inválida son errores semánticos, no fallos de tokenización.

### Operadores

`+ - * **` operan con números; `/` devuelve real aun con enteros; `%` requiere enteros y usa el resto `a - floor(a/b)*b`. División o módulo por cero producen error. Se excluyen resultados complejos y no finitos. `< <= > >=` comparan números. `== !=` comparan numéricamente si ambos operandos admiten conversión; en otro caso comparan cadenas. `eq ne` comparan siempre cadenas.

La conversión booleana admite números finitos (cero falso, otro valor verdadero) y las palabras booleanas en minúsculas del contrato léxico. `&&`, `||` y `!` devuelven `0` o `1`. No se evalúa el hijo derecho de `&&` si el izquierdo es falso, ni el de `||` si el izquierdo es verdadero, **incluidas sus sustituciones de comandos**. `**` tiene la precedencia definida en la sección 5, aunque otras implementaciones de Tcl difieran.

## 15. Programas de aceptación: tareas interesantes

Los siguientes programas son **fuentes de prueba**, no código Python ejecutable. En esta etapa deben producir AST; los resultados indicados serán comprobables cuando exista el intérprete.

### A. Recursión y sustituciones anidadas

```tcl
proc factorial {n} {
    if {$n <= 1} {return 1}
    return [expr {$n * [factorial [expr {$n - 1}]]}]
}
puts [factorial 6]
```

Resultado futuro: `720` (se admite representación numérica equivalente). El cuerpo de `proc` es literal hasta la pasada de análisis de comandos. La expresión del retorno contiene un script que, a su vez, contiene otra expresión.

### B. Estadísticas de una lista

```tcl
set datos [list 4 8 15 16 23 42]
set suma 0
foreach dato $datos {
    set suma [expr {$suma + $dato}]
}
set promedio [expr {$suma / [llength $datos]}]
puts "Promedio: $promedio"
```

Resultado futuro: promedio `18.0`. `datos` es una lista almacenada como valor; `foreach` no analiza su contenido como código.

### C. Histograma con diccionarios

```tcl
set conteo [dict create]
foreach palabra [split {sol luna sol mar luna sol}] {
    if {[dict exists $conteo $palabra]} {
        dict set conteo $palabra [expr {[dict get $conteo $palabra] + 1}]
    } else {
        dict set conteo $palabra 1
    }
}
puts [dict get $conteo sol]
```

Resultado futuro: `3`. Las claves son valores; el comando `dict set` recibe literalmente el nombre de la variable que modifica.

### D. Ciclo `for`, arreglo y errores

```tcl
set tabla(3) 9
for {set i 0} {$i < 5} {incr i} {
    if {$i == 2} {continue}
    puts [expr {$i * $i}]
}
set codigo [catch {expr {10 / 0}} mensaje]
puts "$codigo: $mensaje"
puts $tabla(3)
```

Resultado futuro: cuadrados `0`, `1`, `9`, `16`; código de error `1` y un diagnóstico de división por cero; finalmente `9`.

**Actividad:** elija uno, enumere todas sus invocaciones recursivas de análisis y localice las palabras cuyos contenidos deben analizarse de nuevo.

### Respuesta: invocaciones recursivas de análisis del programa A (factorial)

**Análisis estructural** (`parse_minitcl`):

1. `ScriptLexer.tokenize` → `lex_script` (nivel exterior). Para `proc factorial {n} {...}`: `lex_word` → `lex_bare` ×2, `lex_braced` ×2 (`{n}` y el cuerpo completo: su contenido no se tokeniza todavía).
2. Para `puts [factorial 6]`: `lex_bare` → `lex_bracket` → `lex_script` anidado → `lex_bare` ×2 → vuelve al `]`.
3. `ScriptParser.parse` → `parse_script(EOF)` → `parse_command` ×2 → `parse_word` → `parse_bare_word`/`parse_braced_word`; en `puts`: `parse_part` → `parse_substitution` → `parse_script(RBRACKET)` → `parse_command` → `parse_word` ×2.

**Pasada de análisis de comandos** (`analyze_script`):

4. `cmd_proc` → `parse_list("n")` (parámetros) → `body` → `parse_minitcl(cuerpo, base_offset=…)` (nuevo lexer + parser manual) → `visit_script` con `Context(in_proc=True)`.
5. Dentro del cuerpo, `cmd_if` → `test("$n <= 1")` → `parse_minitcl_expr` (SLY) y `body("return 1")` → `parse_minitcl` → `cmd_return` (válido: `in_proc`).
6. `return [expr {...}]` → `visit_words` visita el `Script` del corchete → `cmd_expr` → `test` → `parse_minitcl_expr("$n * [factorial [expr {$n - 1}]]")`.
7. Dentro de SLY, `[` → `read_substitution` → `ScriptLexer.tokenize_substitution` (que a su vez hace `lex_bracket` anidado para `[expr {$n - 1}]` y `lex_braced` para `{$n - 1}`) → `ScriptParser.parse_substitution` → `parse_script(RBRACKET)` …
8. `visit_expr` encuentra el `SubstitutionExpr` → `visit_script` → `factorial [expr {$n - 1}]` (llamada a un procedimiento: no se reanaliza) → `visit_words` → `cmd_expr` → `parse_minitcl_expr("$n - 1")`.
9. `puts [factorial 6]` → `visit_words` → llamada a `factorial` sin reanálisis.

**Palabras cuyo contenido se vuelve a analizar:** `{n}` (lista de parámetros), el cuerpo de `proc`, `{$n <= 1}` (expresión), `{return 1}` (script), `{$n * [factorial [expr {$n - 1}]]}` (expresión) y `{$n - 1}` (expresión). La celda siguiente analiza los cuatro programas sin ejecutarlos.

In [20]:
PROGRAMS = {
    "A_factorial": """proc factorial {n} {
    if {$n <= 1} {return 1}
    return [expr {$n * [factorial [expr {$n - 1}]]}]
}
puts [factorial 6]""",
    "B_estadisticas": """set datos [list 4 8 15 16 23 42]
set suma 0
foreach dato $datos {
    set suma [expr {$suma + $dato}]
}
set promedio [expr {$suma / [llength $datos]}]
puts "Promedio: $promedio"
""",
    "C_histograma": """set conteo [dict create]
foreach palabra [split {sol luna sol mar luna sol}] {
    if {[dict exists $conteo $palabra]} {
        dict set conteo $palabra [expr {[dict get $conteo $palabra] + 1}]
    } else {
        dict set conteo $palabra 1
    }
}
puts [dict get $conteo sol]""",
    "D_for_catch": """set tabla(3) 9
for {set i 0} {$i < 5} {incr i} {
    if {$i == 2} {continue}
    puts [expr {$i * $i}]
}
set codigo [catch {expr {10 / 0}} mensaje]
puts "$codigo: $mensaje"
puts $tabla(3)""",
}
for name, src in PROGRAMS.items():
    res = analyze_source(src, name)
    print(f"{name}: {len(res.script.commands)} comandos; {len(res.bodies)} cuerpos y "
          f"{len(res.exprs)} expresiones reanalizados")
    for word, expr in res.exprs.items():
        print("    expr", repr(word.parts[0].text))

A_factorial: 2 comandos; 2 cuerpos y 3 expresiones reanalizados
    expr '$n <= 1'
    expr '$n * [factorial [expr {$n - 1}]]'
    expr '$n - 1'
B_estadisticas: 5 comandos; 1 cuerpos y 2 expresiones reanalizados
    expr '$suma + $dato'
    expr '$suma / [llength $datos]'
C_histograma: 3 comandos; 3 cuerpos y 2 expresiones reanalizados
    expr '[dict exists $conteo $palabra]'
    expr '[dict get $conteo $palabra] + 1'
D_for_catch: 5 comandos; 5 cuerpos y 4 expresiones reanalizados
    expr '$i < 5'
    expr '$i == 2'
    expr '$i * $i'
    expr '10 / 0'


In [21]:
# AST estructural completo del programa A (sin offsets).
show_ast(parse_minitcl(PROGRAMS["A_factorial"], "factorial.tcl"))

Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'proc'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'factorial'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'n'
          mode:
            'braced'
        Word
          parts:
            Literal
              text:
                '\n    if {$n <= 1} {return 1}\n    return [expr {$n * [factorial [expr {$n - 1}]]}]\n'
          mode:
            'braced'
    Command
      words:
        Word
          parts:
            Literal
              text:
                'puts'
          mode:
            'bare'
        Word
          parts:
            CommandSubstitution
              script:
                Script
                  commands:
                    Command
                      words:
            

## 16. Pruebas por etapas

No basta con aceptar entradas válidas: compare la **forma** del AST, la cantidad de palabras y el contenido de los literales. Las pruebas del esqueleto se ejecutan ahora. El resto se activa cuando el estudiante implemente las dos fachadas que se indican.

In [22]:
# Pruebas del material suministrado: no requieren completar actividades.
assert ScriptParser([Token("EOF", "", Span(0, 0))]).parse().commands == ()
trailing = [Token("TEXT", "puts", Span(0, 4)),
            Token("WS", " ", Span(4, 5)), Token("EOF", "", Span(5, 5))]
assert len(ScriptParser(trailing).parse().commands[0].words) == 1
try:
    ScriptParser([Token("RBRACKET", "]", Span(0, 1)),
                  Token("EOF", "", Span(1, 1))]).parse()
except (ParseError, NotImplementedError):
    # El esqueleto aún no incorpora todos los diagnósticos.
    pass
else:
    raise AssertionError("No se debe aceptar un cierre exterior")
print("Comprobaciones básicas del esqueleto: correctas.")

Comprobaciones básicas del esqueleto: correctas.


In [23]:
# Fachadas implementadas (mismas firmas): parse_minitcl en minitcl/script_parser.py y
# parse_minitcl_expr en minitcl/expr_parser.py.
from minitcl import parse_minitcl, parse_minitcl_expr

def expr_shape(node):
    """Proyección para comparar estructura sin depender de offsets."""
    if isinstance(node, Constant):
        return node.value
    if isinstance(node, VariableExpr):
        v = node.variable
        return "$" + v.name + (f"({v.index})" if v.index is not None else "")
    if isinstance(node, SubstitutionExpr):
        return "[...]"
    if isinstance(node, Unary):
        return (node.operator, expr_shape(node.operand))
    if isinstance(node, Binary):
        return (node.operator, expr_shape(node.left), expr_shape(node.right))
    raise TypeError(type(node).__name__)

RUN_STUDENT_TESTS = True  # Las dos fachadas están listas.
if RUN_STUDENT_TESTS:
    script = parse_minitcl('puts "Hola $nombre"; set x {a b}')
    assert len(script.commands) == 2
    assert len(script.commands[0].words) == 2
    greeting = script.commands[0].words[1]
    assert greeting.mode == "quoted"
    assert [type(p).__name__ for p in greeting.parts] == ["Literal", "Variable"]
    block = script.commands[1].words[2]
    assert block.mode == "braced" and block.parts[0].text == "a b"
    assert expr_shape(parse_minitcl_expr("2 + 3 * 4")) == ("+", 2, ("*", 3, 4))
    assert expr_shape(parse_minitcl_expr("10 - 3 - 2")) == ("-", ("-", 10, 3), 2)
    assert expr_shape(parse_minitcl_expr("2 ** 3 ** 2")) == ("**", 2, ("**", 3, 2))
    assert expr_shape(parse_minitcl_expr("-2 ** 2")) == ("-", ("**", 2, 2))
    assert expr_shape(parse_minitcl_expr("2 ** -3")) == ("**", 2, ("-", 3))
    assert expr_shape(parse_minitcl_expr("$x != 0 && 10 / $x > 1")) == \
        ("&&", ("!=", "$x", 0), (">", ("/", 10, "$x"), 1))
    print("Pruebas del estudiante: correctas.")

Pruebas del estudiante: correctas.


### Matriz adicional que debe convertir en pruebas

| Entrada | Resultado esperado / nivel |
|---|---|
| `puts ""` y `puts {}` | Dos palabras; argumento vacío, modos diferentes |
| `set x hola_${nombre}_fin` | Tres palabras; la última tiene literal, variable y literal |
| `puts [set x 1; set x]` | Sustitución con dos comandos |
| `puts [list [set x] y]` | Sustituciones anidadas |
| `puts {a {b} c}` | Literal `a {b} c`, sin comandos internos |
| `puts {a\{b\}c}` | Literal conserva las barras y no altera balanceo |
| `puts a#b` / `puts #dato` | `#` dentro del argumento, sin comentario |
| `# comentario` | Script vacío |
| `puts "a; b"` | `;` dentro del literal, sin separar comandos |
| `puts $a(3)` | Referencia al arreglo `a`, índice literal `3` |
| `puts $a($i)` | Fuera del perfil léxico; error localizado |
| `puts {sin cierre` | Error de llave faltante, señalar apertura |
| `puts "sin cierre` | Error de comillas faltantes |
| `puts [set x 1` | Error de corchete faltante |
| `puts {a}b` / `puts "a"b` | Error por texto después de una palabra agrupada |
| `puts ]` | Error: cierre fuera de contexto |
| `set x 1 2` | AST estructural válido; error de aridad en validación |
| `if {1} {puts si} else` | AST estructural válido; falta cuerpo en validación |
| `expr {2 + }` | Script válido; error al analizar su expresión |
| `expr {2 3}` | Error de expresión por residuos |
| `expr {0 && [error inesperado]}` | AST válido; futuro evaluador no ejecuta `error` |
| `proc f {x x} {return $x}` | Error de parámetros duplicados en validación |
| `puts {if mal formado}` | Válido; ese argumento es texto, no un condicional |

Para errores de entrada, las fachadas públicas deben lanzar `ParseError` (incluidos errores del lexer, o una subclase). No capture cualquier `Exception` en las pruebas negativas: ocultaría errores de programación o actividades aún sin resolver.

In [24]:
# Matriz de la sección 16: cada caso indica la etapa en que debe aceptarse o rechazarse.
#   ok          -> estructural y validación correctos
#   lexer/syntax-> ParseError en parse_minitcl
#   expr        -> estructura válida; ParseError al analizar la expresión
#   validation  -> estructura válida; ParseError del validador de comandos
MATRIX = [
    ('puts ""', "ok"), ("puts {}", "ok"), ("set x hola_${nombre}_fin", "ok"),
    ("puts [set x 1; set x]", "ok"), ("puts [list [set x] y]", "ok"),
    ("puts {a {b} c}", "ok"), (r"puts {a\{b\}c}", "ok"), ("puts a#b", "ok"),
    ("puts #dato", "ok"), ("# comentario", "ok"), ('puts "a; b"', "ok"),
    ("puts $a(3)", "ok"), ("puts $a($i)", "lexer"), ("puts {sin cierre", "lexer"),
    ('puts "sin cierre', "lexer"), ("puts [set x 1", "lexer"), ("puts {a}b", "lexer"),
    ('puts "a"b', "lexer"), ("puts ]", "syntax"), ("set x 1 2", "validation"),
    ("if {1} {puts si} else", "validation"), ("expr {2 + }", "expr"),
    ("expr {2 3}", "expr"), ("expr {0 && [error inesperado]}", "ok"),
    ("proc f {x x} {return $x}", "validation"), ("puts {if mal formado}", "ok"),
]

def stage_of(source):
    try:
        script = parse_minitcl(source)
    except ParseError as err:
        return err.stage, err
    try:
        analyze_script(script, source=source)
    except ParseError as err:
        return err.stage, err
    return "ok", None

for source, expected in MATRIX:
    got, err = stage_of(source)
    assert got == expected, (source, expected, got, err)
    print(f"{source!r:38} {got:10} {'' if err is None else err.message}")
print(f"\nMatriz completa: {len(MATRIX)} casos correctos.")

'puts ""'                              ok         
'puts {}'                              ok         
'set x hola_${nombre}_fin'             ok         
'puts [set x 1; set x]'                ok         
'puts [list [set x] y]'                ok         
'puts {a {b} c}'                       ok         
'puts {a\\{b\\}c}'                     ok         
'puts a#b'                             ok         
'puts #dato'                           ok         
'# comentario'                         ok         
'puts "a; b"'                          ok         
'puts $a(3)'                           ok         
'puts $a($i)'                          lexer      Índice de arreglo fuera del perfil: solo se admiten índices literales simples
'puts {sin cierre'                     lexer      Falta '}' que cierre la llave
'puts "sin cierre'                     lexer      Faltan comillas de cierre '"'
'puts [set x 1'                        lexer      Falta ']' que cierre la sustitución de comando
'pu

In [25]:
# Suite completa de pruebas reproducibles (tests/).
import subprocess, sys
completed = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:asyncio"],
                           capture_output=True, text=True)
print(completed.stdout[-1500:], completed.stderr[-1500:])
assert completed.returncode == 0

2m.......................................... [ 43%]
........................................................................ [ 87%]
.....................                                                    [100%]
165 passed in 0.12s
 


## 17. Diagnósticos y estrategia de implementación

Cada diagnóstico debe contener nombre de fuente, línea, columna, elemento esperado y elemento encontrado. El cursor suministrado conserva offsets; **actividad:** calcular línea y columna a partir de la fuente normalizada, y mostrar el fragmento afectado. En un cierre faltante conviene incluir también la posición de apertura.

Orden de trabajo sugerido:

1. Lexer de texto, espacios, separadores y comentarios; comprobar el esqueleto con tokens reales.
2. `BRACED`, variables y palabras con múltiples partes.
3. Comillas y corchetes anidados; probar cambios de contexto y restauración del modo.
4. `ExprLexer(sly.Lexer)` y `ExprParser(sly.Parser)`: tokens, producciones LALR(1), acciones de AST y errores.
5. Validador de formas de comandos y reanálisis de cuerpos y condiciones.
6. Lector de listas, parámetros de procedimientos y casos de integración.
7. Posiciones completas, mensajes y pruebas negativas.

Deténgase en el primer error para esta entrega. La recuperación por sincronización es opcional: no descarte tokens hasta el primer `;` sin considerar comillas y anidamiento.

### Preguntas que deben acompañar la entrega

1. ¿Por qué `set x {$y}` no debe generar una referencia a `y`?
2. ¿Qué evita que `hola_$nombre` se convierta en dos argumentos?
3. ¿Qué parte del sistema distingue `set n 12` de `expr {12}`?
4. ¿Por qué la gramática de SLY admite recursividad izquierda para la suma, mientras la potencia asocia a la derecha?
5. ¿Cómo conserva el análisis de `0 && [error fallo]` la posibilidad de cortocircuito?
6. ¿Qué hace el lexer con `]` dentro de llaves y con `;` dentro de comillas?
7. ¿Qué errores puede detectar el parser genérico y cuáles requieren conocer el comando?
8. ¿Cómo cambian los contextos de `break` y `return` al entrar a un nuevo procedimiento?

### Diagnósticos implementados

`ParseError` contiene el nombre de la fuente, la línea y la columna (calculadas sobre la fuente normalizada con `attach_source`), el elemento esperado (`expected`), el encontrado (`found`), la etapa (`stage`: `lexer`, `syntax`, `expr`, `validation`) y, para cierres faltantes, la posición de apertura (`opening`). `report()` muestra además la línea afectada con `^`. Los errores dentro de cuerpos y condiciones reanalizados se reportan con la posición en la fuente completa, porque el reanálisis propaga `base_offset = word.span.start + 1`.

In [26]:
for bad in ["set a 1\nputs {abc", "proc f {} {\n    set x [expr {1 +}]\n}",
            "puts \"hola [set x\n", "while {$i <} {incr i}", "set x 1 2"]:
    try:
        analyze_source(bad, "demo.tcl")
    except ParseError as err:
        print(f"[{err.stage}] " + err.report())
        print()

[lexer] demo.tcl:2:10: Falta '}' que cierre la llave (esperado: }; encontrado: EOF)
puts {abc
         ^
(apertura en línea 2, columna 6)

[expr] demo.tcl:2:21: Fin inesperado de expresión (esperado: operando; encontrado: fin)
    set x [expr {1 +}]
                    ^

[lexer] demo.tcl:2:1: Falta ']' que cierre la sustitución de comando (esperado: ]; encontrado: EOF)

^
(apertura en línea 1, columna 12)

[expr] demo.tcl:1:12: Fin inesperado de expresión (esperado: operando; encontrado: fin)
while {$i <} {incr i}
           ^

[validation] demo.tcl:1:1: Aridad inválida en 'set': se recibieron 3 argumentos (esperado: set nombre ?valor?; encontrado: 3 argumentos)
set x 1 2
^^^^^^^^^



### Respuestas a las preguntas de la entrega

**1. ¿Por qué `set x {$y}` no debe generar una referencia a `y`?**
Porque las llaves agrupan sin sustituciones. El lexer reconoce `{$y}` como un único token `BRACED` sin examinar su contenido y el parser produce `Word(mode='braced')` con un solo `Literal('$y')`. `set` no es un comando que reinterprete su argumento como script o expresión, así que el analizador tampoco lo reanaliza. Solo las palabras no agrupadas o entre comillas producen nodos `Variable`.

**2. ¿Qué evita que `hola_$nombre` se convierta en dos argumentos?**
Que entre `hola_` y `$nombre` no hay `WS`. El lexer emite `TEXT('hola_')` y `VAR('nombre')` contiguos, y `parse_bare_word` acumula partes mientras el token siguiente esté en FIRST(`part`). Solo un token `WS` separa palabras en `parse_command`. La concatenación de las partes ocurre al ejecutar y no crea argumentos nuevos.

**3. ¿Qué parte del sistema distingue `set n 12` de `expr {12}`?**
El analizador de comandos (`command_analyzer`), porque conoce la forma de cada incorporado. Para el parser estructural ambos son palabras: `12` es `TEXT` y `{12}` es `BRACED`. El lexer de scripts nunca produce `NUMBER`. Solo cuando la pasada de validación reconoce `expr` entrega el contenido de la llave a `parse_minitcl_expr`. Entonces `ExprLexer` genera `NUMBER` y el parser SLY produce `Constant(12)`. En `set n 12`, el `12` sigue siendo un `Literal`.

**4. ¿Por qué la gramática de SLY admite recursividad izquierda para la suma, mientras la potencia asocia a la derecha?**
SLY genera un analizador LALR(1) ascendente: reduce con una pila de estados y nunca "llama" a `additive` antes de consumir entrada, así que la recursividad izquierda no provoca recursión infinita y además da asociación izquierda (`(10-3)-2`). La potencia asocia a la derecha porque su recursión está en el lado derecho: `power : primary POWER unary` y `unary` deriva otra vez `power`. El izquierdo es un `primary`, que no puede contener `**`.

**5. ¿Cómo conserva el análisis de `0 && [error fallo]` la posibilidad de cortocircuito?**
El lexer SLY convierte `[error fallo]` en un único token `CMD_SUBST` cuyo valor es el `Script` producido por el parser manual. Ese script **no se ejecuta**. El AST es `Binary('&&', Constant(0), SubstitutionExpr(script))`. El evaluador futuro evalúa primero el operando izquierdo y, si es falso, no evalúa el derecho ni su sustitución. Si el análisis ejecutara la sustitución, el cortocircuito se perdería.

**6. ¿Qué hace el lexer con `]` dentro de llaves y con `;` dentro de comillas?**
Dentro de llaves, `lex_braced` solo cuenta `{` y `}` (saltando `\{` y `\}`), así que `]` forma parte del valor del `BRACED` y no cierra ninguna sustitución (ej. `[puts {]}]`). Dentro de comillas, `lex_quoted` añade `;` al `TEXT` actual: no emite `SEP` ni separa comandos (`puts "a; b"` es un único comando).

**7. ¿Qué errores puede detectar el parser genérico y cuáles requieren conocer el comando?**
*Genéricos (lexer + parser estructural):* llaves, comillas o corchetes sin cerrar (con su apertura), texto después de una palabra agrupada, `{`/`"` no escapados dentro de una palabra no agrupada, `]` fuera de contexto, variables mal formadas o con índice fuera del perfil (`$a($i)`), escapes desconocidos y la continuación barra + salto.
*Requieren conocer el comando:* aridad (`set x 1 2`), formas incompletas (`if {1} {puts si} else`), errores de sintaxis de expresiones (solo se sabe que `{2 + }` es expresión porque lo recibe `expr`), parámetros duplicados o inválidos de `proc`, cuerpos y condiciones que no están entre llaves, nombres de comando calculados, redefinición de incorporados, subcomandos de `dict`/`string` y `return`/`break`/`continue` fuera de contexto.

**8. ¿Cómo cambian los contextos de `break` y `return` al entrar a un nuevo procedimiento?**
El analizador lleva un `Context(in_proc, in_loop)`. Los cuerpos de `while`, `for` y `foreach` se analizan con `in_loop=True` y conservan `in_proc`. El cuerpo de `proc` empieza un contexto nuevo: `Context(in_proc=True, in_loop=False)`. Por eso un `break` dentro de un procedimiento definido dentro de un ciclo es un error: no puede escapar hacia el ciclo del llamador. `return` exige `in_proc=True`. Al ejecutar, cada llamada creará además un ámbito local nuevo, y `global` enlazará nombres con el entorno global.

## 18. Entregables y criterios de evaluación

Entregar el cuaderno completado, los módulos equivalentes (`model.py`, `script_lexer.py`, `script_parser.py`, `expr_lexer.py`, `expr_parser.py`, `command_analyzer.py`) y pruebas reproducibles. Los nombres de código se escriben en inglés y las explicaciones en español.

| Criterio | Peso |
|---|---:|
| Gramática aplicada y explicación de decisiones predictivas | 15 % |
| Léxico contextual, balanceo y límites de palabras | 20 % |
| Parser estructural y construcción correcta del AST | 25 % |
| Lexer y parser con SLY: AST, precedencia y asociatividad | 20 % |
| Validación de comandos y análisis de cuerpos estáticos | 10 % |
| Diagnósticos y pruebas positivas/negativas | 10 % |

**Condición de terminación:** no quedan `NotImplementedError` en las rutas obligatorias, se puede analizar cada programa de la sección 15 sin ejecutarlo, se obtienen AST verificables y se rechazan los casos inválidos en la etapa apropiada. El intérprete y el generador de código serán trabajos posteriores.

La entrega debe demostrar que `ExprLexer` deriva de `sly.Lexer` y `ExprParser` de `sly.Parser`, que la gramática completa genera tablas sin conflictos no justificados y que sus acciones producen AST. Debe comprobarse que tanto `expr` como las condiciones pasan por la misma fachada SLY. Los scripts se analizan con el lexer manual y descenso recursivo.


## 19. Referencias y decisiones de diseño

Fuentes primarias consultadas para delimitar el lenguaje:

- [Reglas de sintaxis de Tcl](https://www.tcl-lang.org/man/tcl8.7/TclCmd/Tcl.html): separación entre comandos, palabras, agrupación y sustituciones.
- [Picol de antirez](https://github.com/antirez/picol): referencia de intérprete pequeño y didáctico. Existen versiones y derivados con capacidades distintas; no se asume que todos tengan el mismo repertorio.
- [Jim Tcl](https://jim.tcl-lang.org/home/doc/www/www/index.html) y [su documentación](https://jim.tcl-lang.org/home/doc/trunk/jim_tcl.txt): referencia de una implementación con un alcance considerablemente mayor.

La gramática y los contratos de este cuaderno son una **propuesta docente propia**, no una transcripción del manual de Tcl ni una promesa de compatibilidad completa. Los límites de índices, escapes, cuerpos estáticos, `expr`, listas y `catch` son decisiones de este perfil. El tamaño intermedio se busca por el alcance de la biblioteca y la arquitectura; no por una cifra de líneas de código.

Este cuaderno retoma el alcance y la precedencia descritos en la especificación MiniTcl 1.0 del curso y documenta explícitamente sus ampliaciones. Las fuentes externas sirven para contextualizar; el contrato obligatorio para el estudiante es el definido aquí.

- [Documentación oficial de SLY](https://sly.readthedocs.io/en/latest/sly.html): clases `Lexer`/`Parser`, producciones, posiciones y generación LALR(1).


## 20. EBNF para diagramas railroad: dos entradas y un puente

El archivo `MiniTcl_Railroad.ebnf` usa notación de estilo W3C para el generador de Bottlecaps. `manual_script_input` inicia la gramática del lexer manual y el parser descendente recursivo. `sly_expression_input` inicia la gramática LALR(1) de SLY; su fin de entrada es implícito.

Los diagramas de `builtin_forms` son contratos de validación, no alternativas nuevas del parser estructural. `expr_argument`, `test` y `body` aparecen como `BRACED`: el análisis de comandos entrega el contenido de los dos primeros a SLY y el de `body` al parser manual. Es una delegación entre analizadores, no la inserción de la gramática de expresiones dentro del lexer de scripts.

`CMD_SUBST` representa un token SLY cuyo valor es un AST `Script`; el patrón de caracteres `[...]` se muestra en `substitution`, dentro de la gramática manual. Las reglas auxiliares `variable_syntax`, `number_syntax` y `boolean_syntax` explican la forma del texto; no son producciones adicionales de SLY ni sustituyen sus expresiones regulares. Los nombres de tokens entre comillas son etiquetas para el diagrama.

Las producciones de expresiones conservan recursividad izquierda intencionadamente: son para SLY. Las de scripts conservan repeticiones para facilitar el descenso recursivo.

Copie el siguiente bloque en https://www.bottlecaps.de/rr/ui:

```ebnf
manual_script_input ::= program
program ::= script 'EOF'

script ::= ('WS' | 'SEP')* (command (command_gap command)*)? ('WS' | 'SEP')*
command_gap ::= 'WS'* 'SEP' ('WS' | 'SEP')*
command ::= word ('WS' word)*
word ::= 'BRACED' | quoted_word | bare_word
quoted_word ::= 'QOPEN' part* 'QCLOSE'
bare_word ::= part+
part ::= 'TEXT' | 'VAR' | substitution
substitution ::= '[' script ']'

sly_expression_input ::= expression
expression ::= logical_or
logical_or ::= logical_or '||' logical_and | logical_and
logical_and ::= logical_and '&&' equality | equality
equality ::= equality ('==' | '!=' | 'eq' | 'ne') relational | relational
relational ::= relational ('<' | '<=' | '>' | '>=') additive | additive
additive ::= additive ('+' | '-') multiplicative | multiplicative
multiplicative ::= multiplicative ('*' | '/' | '%') unary | unary
unary ::= ('+' | '-' | '!') unary | power
power ::= primary '**' unary | primary
primary ::= 'NUMBER' | 'BOOL' | 'VAR' | 'STRING' | '(' expression ')' | 'CMD_SUBST'

builtin_forms ::= set_cmd | unset_cmd | incr_cmd | append_cmd | puts_cmd
                | expr_cmd | if_cmd | while_cmd | for_cmd | foreach_cmd
                | proc_cmd | return_cmd | break_cmd | continue_cmd
                | global_cmd | catch_cmd | error_cmd
                | list_cmd | llength_cmd | lindex_cmd | lappend_cmd | lset_cmd
                | dict_cmd | string_cmd | split_cmd | join_cmd

set_cmd ::= 'set' 'WS' name ('WS' word)?
unset_cmd ::= 'unset' 'WS' name
incr_cmd ::= 'incr' 'WS' name ('WS' word)?
append_cmd ::= 'append' 'WS' name ('WS' word)+
puts_cmd ::= 'puts' 'WS' ('-nonewline' 'WS')? word
expr_cmd ::= 'expr' 'WS' expr_argument
if_cmd ::= 'if' 'WS' test ('WS' 'then')? 'WS' body
           ('WS' 'elseif' 'WS' test ('WS' 'then')? 'WS' body)*
           ('WS' 'else' 'WS' body)?
while_cmd ::= 'while' 'WS' test 'WS' body
for_cmd ::= 'for' 'WS' body 'WS' test 'WS' body 'WS' body
foreach_cmd ::= 'foreach' 'WS' identifier 'WS' word 'WS' body
proc_cmd ::= 'proc' 'WS' identifier 'WS' parameters 'WS' body
return_cmd ::= 'return' ('WS' word)?
break_cmd ::= 'break'
continue_cmd ::= 'continue'
global_cmd ::= 'global' ('WS' identifier)+
catch_cmd ::= 'catch' 'WS' body ('WS' identifier)?
error_cmd ::= 'error' 'WS' word
call_cmd ::= word ('WS' word)*

body ::= 'BRACED'
expr_argument ::= 'BRACED'
test ::= expr_argument
parameters ::= 'BRACED'
parameter_content ::= 'LIST_WS'* (identifier ('LIST_WS' identifier)*)? 'LIST_WS'*
name ::= identifier ('(' index ')')?

list_cmd ::= 'list' ('WS' word)*
llength_cmd ::= 'llength' 'WS' word
lindex_cmd ::= 'lindex' 'WS' word 'WS' word
lappend_cmd ::= 'lappend' 'WS' name ('WS' word)*
lset_cmd ::= 'lset' 'WS' name 'WS' word 'WS' word
dict_cmd ::= dict_create_cmd | dict_get_cmd | dict_exists_cmd | dict_set_cmd
dict_create_cmd ::= 'dict' 'WS' 'create' ('WS' word 'WS' word)*
dict_get_cmd ::= 'dict' 'WS' 'get' 'WS' word 'WS' word
dict_exists_cmd ::= 'dict' 'WS' 'exists' 'WS' word 'WS' word
dict_set_cmd ::= 'dict' 'WS' 'set' 'WS' name 'WS' word 'WS' word
string_cmd ::= string_length_cmd | string_case_cmd | string_compare_cmd
string_length_cmd ::= 'string' 'WS' 'length' 'WS' word
string_case_cmd ::= 'string' 'WS' ('tolower' | 'toupper') 'WS' word
string_compare_cmd ::= 'string' 'WS' 'compare' 'WS' word 'WS' word
split_cmd ::= 'split' 'WS' word ('WS' word)?
join_cmd ::= 'join' 'WS' word ('WS' word)?

list_value ::= 'LIST_WS'* (element ('LIST_WS' element)*)? 'LIST_WS'*
element ::= 'LIST_BRACED' | 'LIST_QUOTED' | 'LIST_BARE'

variable_syntax ::= '$' identifier ('(' index ')')? | '${' identifier '}'
identifier ::= letter_or_underscore (letter_or_underscore | digit)*
letter_or_underscore ::= [A-Z] | [a-z] | '_'
digit ::= [0-9]
index ::= index_char+
index_char ::= letter_or_underscore | digit | '.' | '-' | ':'
number_syntax ::= (digit+ ('.' digit*)? | '.' digit+) exponent?
exponent ::= ('e' | 'E') ('+' | '-')? digit+
boolean_syntax ::= 'true' | 'false' | 'yes' | 'no' | 'on' | 'off'
```